<a href="https://colab.research.google.com/github/IgorKovacevicENNOH/ENNOH_Modelling/blob/main/Intermediate_to_model_data_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

In [ ]:
! pip install pypsa highspy openpyxl "xarray<=2024.9.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 392.0/392.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.3/212.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.6 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
  Attempting uninstall: xarray
    Found existing installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
# Import packages
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
import json
import time
import pypsa
import warnings

ROOT_DIR = os.getcwd()
PROJECT_DIR = os.path.join(ROOT_DIR, "drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model")


In [ ]:
sys.path.append(PROJECT_DIR)

from modules.getting_input_data import get_input_data

input_file_name = "input_file.xlsx"
input_data = get_input_data(PROJECT_DIR, input_file_name)

File found at: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/input_file.xlsx
The input data has been imported.


In [ ]:
input_data

{'project_name': 'Europe',
 'regions': '["Albania", "Austria", "Belgium", "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Cyprus", "Czechia", "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta", "Montenegro", "Netherlands", "North Macedonia", "Norway", "Poland", "Portugal", "Romania", "Serbia", "Slovakia", "Slovenia", "Spain", "Sweden", "Switzerland", "United Kingdom"]',
 'zones': '["AL00", "AT00", "BA00", "BE00", "BG00", "CH00", "CY00", "CZ00", "DE00", "DKE1", "DKW1", "EE00", "ES00", "FI00", "FR00", "GR00", "GR03", "HR00", "HU00", "IE00", "ITCA", "ITCN", "ITCS", "ITN1", "ITS1", "ITSA", "ITSI", "LT00", "LUF1", "LUG1", "LUV1", "LV00", "MD00", "ME00", "MK00", "MT00", "NL00", "NOM1", "NON1", "NOS1", "NOS2", "NOS3", "PL00", "PT00", "RO00", "RS00", "SE01", "SE02", "SE03", "SE04", "SI00", "SK00", "TR00", "UA00", "UK00", "UKNI"]',
 'data_set': 'TYNDP_scenario_2026',
 'year': 2035,
 'scenario': 'DE',
 'we

In [ ]:
zones = json.loads(input_data["zones"])
year = input_data["year"]
scenario =  input_data["scenario"]
project_name =  input_data["project_name"]

DATA_DIR = os.path.join(PROJECT_DIR, str(input_data["data_set"]),input_data["raw_data_dir"])

INTER_DIR = os.path.join(PROJECT_DIR, str(input_data["data_set"]),input_data["inter_dir"],input_data["project_name"], input_data["scenario"], str(input_data["year"]))
INTER_DIR_PROFILE = os.path.join(PROJECT_DIR, str(input_data["data_set"]), input_data['inter_dir'], input_data["project_name"], input_data["scenario"], str(input_data["year"]), f'profile_{input_data["weather_profile"]}')

MODEL_DATA_DIR = os.path.join(PROJECT_DIR,str(input_data["data_set"]),input_data['model_data_dir'],input_data["project_name"],input_data["scenario"], str(input_data["year"]), f'profile_{input_data["weather_profile"]}')
MODEL_DATA_FIX = os.path.join(PROJECT_DIR,str(input_data["data_set"]),input_data['model_data_dir'])


capacities = {}

In [ ]:
import json
import pandas as pd

# Extract the regional mapping from the input_data
country_names = json.loads(input_data['regions'])

# Correct manual mapping to ensure data integrity with the notebook's acronyms
country_mapping = {
    'Albania': 'AL', 'Austria': 'AT', 'Belgium': 'BE', 'Bosnia and Herzegovina': 'BA',
    'Bulgaria': 'BG', 'Croatia': 'HR', 'Cyprus': 'CY', 'Czechia': 'CZ',
    'Denmark': 'DK', 'Estonia': 'EE', 'Finland': 'FI', 'France': 'FR',
    'Germany': 'DE', 'Greece': 'GR', 'Hungary': 'HU', 'Ireland': 'IE',
    'Italy': 'IT', 'Latvia': 'LV', 'Lithuania': 'LT', 'Luxembourg': 'LU',
    'Malta': 'MT', 'Montenegro': 'ME', 'Netherlands': 'NL', 'North Macedonia': 'MK',
    'Norway': 'NO', 'Poland': 'PL', 'Portugal': 'PT', 'Romania': 'RO',
    'Serbia': 'RS', 'Slovakia': 'SK', 'Slovenia': 'SI', 'Spain': 'ES',
    'Sweden': 'SE', 'Switzerland': 'CH', 'United Kingdom': 'UK'
}

# Filter only the countries present in the current project
country_to_acronym = {name: country_mapping[name] for name in country_names if name in country_mapping}

display(pd.Series(country_to_acronym, name='Acronym'))

Albania                   AL
Austria                   AT
Belgium                   BE
Bosnia and Herzegovina    BA
Bulgaria                  BG
Croatia                   HR
Cyprus                    CY
Czechia                   CZ
Denmark                   DK
Estonia                   EE
Finland                   FI
France                    FR
Germany                   DE
Greece                    GR
Hungary                   HU
Ireland                   IE
Italy                     IT
Latvia                    LV
Lithuania                 LT
Luxembourg                LU
Malta                     MT
Montenegro                ME
Netherlands               NL
North Macedonia           MK
Norway                    NO
Poland                    PL
Portugal                  PT
Romania                   RO
Serbia                    RS
Slovakia                  SK
Slovenia                  SI
Spain                     ES
Sweden                    SE
Switzerland               CH
United Kingdom

# Reading Itermediate_data Folder

Most data is imported from the intermediate_data folder, specifically:
*   Generation capacities
*   RES profiles
*   Hydrogen data
*   Demand profiles

The only exception is NTC data - electricity and hydrogen, which is sourced directly from the raw data folder.

## 1, Capacities

In [ ]:
# Define the path to the capacities file
capacities_file_path = os.path.join(INTER_DIR, f"capacity.json")

# Load the capacities data
try:
    with open(capacities_file_path, 'r') as f:
        capacities = json.load(f)
    print(f"✅ Successfully loaded capacities for {project_name} project in {year}.")
    zones = list(capacities.keys())
    print(f"    Zones are:", zones)
except FileNotFoundError:
    print(f"❌ Error: The file {os.path.basename(capacities_file_path)} was not found. Please ensure it's in the correct directory ({os.path.join(PROJECT_DIR, project_name)}).")
    # Diagnostic: List files in the project directory to help the user
    print(f"Files in {os.path.join(PROJECT_DIR, project_name)}:")
    try:
        for item in os.listdir(os.path.join(PROJECT_DIR, project_name)):
            print(f"  - {item}")
    except Exception as e:
        print(f"  Could not list directory contents: {e}")
except json.JSONDecodeError:
    print(f"❌ Error: Could not decode JSON from {os.path.basename(capacities_file_path)}. Check file format.")
except Exception as e:
    print(f"❌ An unexpected error occurred while loading capacities: {e}")

✅ Successfully loaded capacities for Europe project in 2035.
    Zones are: ['AL00', 'AT00', 'BA00', 'BE00', 'BG00', 'CH00', 'CY00', 'CZ00', 'DE00', 'DKE1', 'DKW1', 'EE00', 'ES00', 'FI00', 'FR00', 'GR00', 'GR03', 'HR00', 'HU00', 'IE00', 'ITCA', 'ITCN', 'ITCS', 'ITN1', 'ITS1', 'ITSA', 'ITSI', 'LT00', 'LUF1', 'LUG1', 'LUV1', 'LV00', 'MD00', 'ME00', 'MK00', 'MT00', 'NL00', 'NOM1', 'NON1', 'NOS1', 'NOS2', 'NOS3', 'PL00', 'PT00', 'RO00', 'RS00', 'SE01', 'SE02', 'SE03', 'SE04', 'SI00', 'SK00', 'TR00', 'UA00', 'UK00', 'UKNI']


## 1,a Offshore Capacities

In [ ]:
# Define the path to the offshore capacities file
off_capacities_file_path = os.path.join(INTER_DIR, f"capacities_offshore.json")

# Load the offshore capacities data
try:
    with open(off_capacities_file_path, 'r') as f:
        offshore_capacities = json.load(f)
    print(f"✅ Successfully loaded offshore capacities for {project_name} project in {year}.")
    offshore_regions = list(offshore_capacities.keys())
    print(f"    Offshore regions are:", offshore_regions)
except FileNotFoundError:
    print(f"❌ Error: The file {os.path.basename(off_capacities_file_path)} was not found. Please ensure it's in the correct directory ({os.path.join(PROJECT_DIR, project_name)}).")
    # Diagnostic: List files in the project directory to help the user
    print(f"Files in {os.path.join(PROJECT_DIR, project_name)}:")
    try:
        for item in os.listdir(os.path.join(PROJECT_DIR, project_name)):
            print(f"  - {item}")
    except Exception as e:
        print(f"  Could not list directory contents: {e}")
except json.JSONDecodeError:
    print(f"❌ Error: Could not decode JSON from {os.path.basename(off_capacities_file_path)}. Check file format.")
except Exception as e:
    print(f"❌ An unexpected error occurred while loading offshore capacities: {e}")

✅ Successfully loaded offshore capacities for Europe project in 2035.
    Offshore regions are: ['wind', 'electrolyser']


## 2, Electricity

### RES Profiles

In [ ]:
# Define the path to the RES profiles file
res_profiles_path = os.path.join(INTER_DIR_PROFILE, f"RES_profiles.csv")

# Load the data and parse datetime in column 0
try:
    RES_profiles = pd.read_csv(res_profiles_path, index_col=0, parse_dates=True)
    print(f"✅ Successfully loaded RES profiles. Shape: {RES_profiles.shape}")
except FileNotFoundError:
    print(f"❌ Error: The file {os.path.basename(res_profiles_path)} was not found.\nCheck if it exists in {os.path.dirname(res_profiles_path)}.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ Successfully loaded RES profiles. Shape: (8760, 459)


### Other RES and Non RES

In [ ]:
print(INTER_DIR)

/content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/intermediate_data/Europe/DE/2035


In [ ]:
import os
import pandas as pd

# Define the path to the Other profiles file
other_res_path = os.path.join(INTER_DIR, "Other_profiles.csv")

# Load the data and parse datetime in column 0
try:
    Other_df = pd.read_csv(other_res_path, index_col=0, parse_dates=True)
    print(f"✅ Successfully loaded Other profiles. Shape: {Other_df.shape}")
except FileNotFoundError:
    print(f"❌ Error: The file {os.path.basename(other_res_path)} was not found in {INTER_DIR}.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")


✅ Successfully loaded Other profiles. Shape: (8760, 112)


### Offshore Profiles

In [ ]:
import os
import pandas as pd

# Define the path to the Offshore profiles file
offshore_profiles_path = os.path.join(INTER_DIR_PROFILE, "Offshore_profiles.csv")

# Load the data
try:
    Offshore_profiles_df = pd.read_csv(offshore_profiles_path, index_col=0, parse_dates=True)
    print(f"✅ Successfully loaded Offshore profiles. Shape: {Offshore_profiles_df.shape}")
except FileNotFoundError:
    print(f"❌ Error: The file Offshore_profiles.csv was not found in {INTER_DIR}.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ Successfully loaded Offshore profiles. Shape: (8760, 54)



## 3, Hydrogen Data - H2 data



In [ ]:
import os
import json

# Define the path to the combined H2 data file
H2_data_path = os.path.join(INTER_DIR, f"H2_data.json")

# Load the H2 data
try:
    with open(H2_data_path, 'r') as f:
        H2_data = json.load(f)
    print(f"✅ Successfully loaded H2 data from: {os.path.basename(H2_data_path)}")
except FileNotFoundError:
    print(f"❌ Error: The file {os.path.basename(H2_data_path)} was not found in {os.path.dirname(H2_data_path)}.")
except json.JSONDecodeError:
    print(f"❌ Error: Could not decode JSON from {os.path.basename(H2_data_path)}. Check file format.")
except Exception as e:
    print(f"❌ An unexpected error occurred while loading H2 data: {e}")


✅ Successfully loaded H2 data from: H2_data.json


### SMR

In [ ]:
# Extract SMR capacity data from the combined h2_data dictionary
try:
    if 'SMR_data' in H2_data:
        SMR = pd.DataFrame(H2_data['SMR_data'])
        # Assuming 'NODE' is the first column, set it as the index to match index_col=0
        if 'NODE' in SMR.columns:
            SMR.set_index('NODE', inplace=True)

        print("✅ Successfully extracted SMR data from H2 data.")
    else:
        print("❌ Error: 'SMR_data' key not found in H2 data.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ Successfully extracted SMR data from H2 data.


### H2 storage

In [ ]:
# Extract H2 storage data from the combined h2_data dictionary
try:
    if 'Storage_data' in H2_data:
        H2_storage = pd.DataFrame(H2_data['Storage_data'])

        print("✅ Successfully extracted H2 storage data from combined H2 data.")
    else:
        print("❌ Error: 'Storage_data' key not found in h2_data.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ Successfully extracted H2 storage data from combined H2 data.


In [ ]:
print(f"Total rows in SMR DataFrame: {len(SMR) if 'SMR' in locals() else 0}")
print(f"Total rows in H2_storage DataFrame: {len(H2_storage) if 'H2_storage' in locals() else 0}")

if 'SMR' in locals():
    unique_smr_nodes = SMR.index.nunique()
    print(f"Unique H2 nodes in SMR: {unique_smr_nodes}")

if 'H2_storage' in locals():
    unique_storage_nodes = H2_storage['NODE'].nunique()
    print(f"Unique H2 nodes in H2_storage: {unique_storage_nodes}")


Total rows in SMR DataFrame: 45
Total rows in H2_storage DataFrame: 46
Unique H2 nodes in SMR: 45
Unique H2 nodes in H2_storage: 45


In [ ]:
H2_data['Import_data']

[{'YEAR': 2035,
  'SCENARIO': 'All',
  'CORRIDOR': 'TN-ITh2-LOW',
  'NODE FROM': 'TN',
  'NODE TO': 'ITh2',
  'MAX CAPACITY [MW]': 'TIME-SERIES-DATA',
  'OFFER QUANTITY [MW]': 'TIME-SERIES-DATA',
  'OFFER PRICE [€/MWh]': 0.0,
  'MAX ENERGY YEAR [GWh]': nan,
  'Type': 'Pipeline',
  'Fuel': 'Pure Hydrogen',
  'CORRIDOR_YEAR': 'TN-ITh2-LOW-2035',
  'RAMP UP [MW/min]': nan,
  'RAMP DOWN [MW/min]': nan,
  'Unnamed: 14': nan},
 {'YEAR': 2035,
  'SCENARIO': 'All',
  'CORRIDOR': 'MA-ESh2-LOW',
  'NODE FROM': 'MA',
  'NODE TO': 'ESh2',
  'MAX CAPACITY [MW]': 'TIME-SERIES-DATA',
  'OFFER QUANTITY [MW]': 'TIME-SERIES-DATA',
  'OFFER PRICE [€/MWh]': 0.0,
  'MAX ENERGY YEAR [GWh]': nan,
  'Type': 'Pipeline',
  'Fuel': 'Pure Hydrogen',
  'CORRIDOR_YEAR': 'MA-ESh2-LOW-2035',
  'RAMP UP [MW/min]': nan,
  'RAMP DOWN [MW/min]': nan,
  'Unnamed: 14': nan},
 {'YEAR': 2035,
  'SCENARIO': 'All',
  'CORRIDOR': 'UA-IB_SKh2E-LOW',
  'NODE FROM': 'UA',
  'NODE TO': 'IB_SKh2E',
  'MAX CAPACITY [MW]': 'TIME-SERIE

### Import H2

In [ ]:
# Extract H2 import data from the combined h2_data dictionary
try:
    if 'Import_data' in H2_data:
        H2_imports = pd.DataFrame(H2_data['Import_data'])

        # Set NODE as index if it exists to match previous logic
        if 'NODE' in H2_imports.columns:
            H2_imports.set_index('NODE', inplace=True)

        print(f"✅ Successfully extracted H2 import data from H2_data.")
    else:
        print("❌ Error: 'Import_data' key not found in H2_data.")

except Exception as e:
    print(f"❌ An unexpected error occurred while extracting H2 imports: {e}")

✅ Successfully extracted H2 import data from H2_data.


In [ ]:
import pandas as pd

# Assuming h2_imports dataframe is already loaded
if 'H2_imports' in locals() and not H2_imports.empty:
    # Extract relevant columns to create H2_imports_df
    Import_NTC_H2 = H2_imports[['NODE FROM', 'NODE TO', 'OFFER QUANTITY [MW]', 'OFFER PRICE [€/MWh]', 'Type', 'Fuel']].copy()

    # Rename columns as requested
    Import_NTC_H2.columns = ['zone from', 'zone to', 'capacity', 'price', 'type', 'fuel']

    print("✅ Successfully created Import_NTC_H2.")
else:
    print("❌ Error: h2_imports dataframe is not defined or is empty. Initializing Import_NTC_H2 as empty DataFrame.")
    Import_NTC_H2 = pd.DataFrame(columns=['zone from', 'zone to', 'capacity', 'price', 'type', 'fuel'])

✅ Successfully created Import_NTC_H2.


In [ ]:
# filtering import pipeline and ammonia
if not Import_NTC_H2.empty:
    # Group by 'zone from' and 'zone to', then select the row with the highest price
    Import_NTC_H2_filtered = Import_NTC_H2.loc[Import_NTC_H2.groupby(['zone from', 'zone to'])['price'].idxmax()].reset_index(drop=True)
    print("✅ Filtered Import_NTC_H2 to keep one pipeline per unique 'zone from' and 'zone to' pair, choosing the one with the higher price.")
    # Update Import_NTC_H2 to the filtered version for subsequent use
    Import_NTC_H2 = Import_NTC_H2_filtered
else:
    print("❌ Import_NTC_H2 DataFrame is empty, no filtering performed.")

✅ Filtered Import_NTC_H2 to keep one pipeline per unique 'zone from' and 'zone to' pair, choosing the one with the higher price.


## 3,a H2 import profiles

In [ ]:
import os
import pandas as pd

# Define the path to the H2 import profiles file
h2_import_profiles_path = os.path.join(INTER_DIR, "H2_import_profiles.csv")

# Load the data and parse datetime in column 0
try:
    H2_import_profiles = pd.read_csv(h2_import_profiles_path, index_col=0, parse_dates=True)
    print(f"✅ Successfully loaded H2 import profiles. Shape: {H2_import_profiles.shape}")
except FileNotFoundError:
    print(f"❌ Error: The file {os.path.basename(h2_import_profiles_path)} was not found.\nCheck if it exists in {os.path.dirname(h2_import_profiles_path)}.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ Successfully loaded H2 import profiles. Shape: (8760, 4)


In [ ]:
H2_import_profiles.head()

,MA-ESh2-LOW-2035,TN-ITh2-LOW-2035,DZ-ITh2-LOW-2035,UA-IB_SKh2E-LOW-2035
2035-01-01 00:00:00,0,3382.988469,1409.131936,1424.732861
2035-01-01 01:00:00,0,3382.988469,1409.131936,1424.732861
2035-01-01 02:00:00,0,3382.988469,1409.131936,1424.732861
2035-01-01 03:00:00,0,3382.988469,1409.131936,1424.732861
2035-01-01 04:00:00,0,3382.988469,1409.131936,1424.732861


## 4, Demand Profiles

In [ ]:
# Define the path to the demand profiles file
demand_profiles_path = os.path.join(INTER_DIR_PROFILE,f"demand_profiles.csv")

# Load the data and parse datetime in column 0
try:
    demand_profiles = pd.read_csv(demand_profiles_path, index_col=0, parse_dates=True)
    print(f"✅ Successfully loaded demand profiles. Shape: {demand_profiles.shape}")
except FileNotFoundError:
    print(f"❌ Error: The file {os.path.basename(demand_profiles_path)} was not found.\nCheck if it exists in {os.path.dirname(demand_profiles_path)}.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ Successfully loaded demand profiles. Shape: (8760, 373)


In [ ]:
global_columns = [col for col in demand_profiles.columns if col.startswith('Global')]
print(f"Columns starting with 'Global': {global_columns}")

Columns starting with 'Global': ['Global_sng', 'Global_e-diesel', 'Global_e-kerosene']


In [ ]:
# Identifying the correct column names from the previous check: 'Global_sng', 'Global_e-diesel', 'Global_e-kerosene'
target_map = {
    'Global_sng': 'SNG',
    'Global_e-diesel': 'eDiesel',
    'Global_e-kerosene': 'eKerosine'
}

cols_to_extract = [c for c in target_map.keys() if c in demand_profiles.columns]

if cols_to_extract:
    # 1 & 2. Form Global_efuels_df and rename columns
    Global_efuels_df = demand_profiles[cols_to_extract].copy()
    Global_efuels_df.rename(columns=target_map, inplace=True)

    # 3. Delete these columns from demand profiles
    demand_profiles.drop(columns=cols_to_extract, inplace=True)

    print("✅ Successfully created Global_efuels_df and removed them from demand_profiles.")
    display(Global_efuels_df.head())
else:
    # Initialize empty if not found to avoid NameErrors later
    Global_efuels_df = pd.DataFrame()
    print("⚠️ Specified Global columns not found in demand_profiles.")

✅ Successfully created Global_efuels_df and removed them from demand_profiles.


,SNG,eDiesel,eKerosine
2035-01-01 00:00:00,2416.94,31063.53,0.0
2035-01-01 01:00:00,2416.94,31063.53,0.0
2035-01-01 02:00:00,2416.94,31063.53,0.0
2035-01-01 03:00:00,2416.94,31063.53,0.0
2035-01-01 04:00:00,2416.94,31063.53,0.0


In [ ]:
h2_non_heat_cols = [col for col in demand_profiles.columns if 'h2' in col.lower() and 'heat' not in col.lower()]
print(f"Found {len(h2_non_heat_cols)} columns matching criteria:")
print(h2_non_heat_cols)

Found 45 columns matching criteria:
['ALh2', 'ATh2', 'BAh2', 'BEh2', 'BGh2', 'CHh2', 'CYh2', 'CZh2', 'DEh2', 'DEh2Z1', 'DKh2', 'EEh2', 'ESh2', 'FIh2', 'FIh2Al', 'FIh2N', 'FIh2S', 'FRh2', 'FRh2N', 'FRh2S', 'FRh2SW', 'GRh2', 'HRh2', 'HUh2', 'IEh2', 'ITh2', 'LTh2', 'LTh2Z1', 'LUh2', 'LVh2', 'MDh2', 'MKh2', 'MTh2', 'NLh2', 'NOh2', 'PLh2', 'PTh2', 'PTh2Z1', 'ROh2', 'RSh2', 'SEh2', 'SIh2', 'SKh2E', 'SKh2W', 'UKh2']


## 5, Net Transfer Capacity - Electricity and Hydrogen

NTC data is imported from NTC dictionary prepared in the file: Reading_NTCs_Offshore.ipynb:

Electricity and Hydrogen data

In [ ]:
import os
import json

# Define the path to NTC_dic.json
NTC_path = os.path.join(INTER_DIR, 'NTC_dic.json')

# Load the JSON data
try:
    with open(NTC_path, 'r') as f:
        NTC_dic = json.load(f)
    print(f"✅ Successfully loaded NTC_dic from {NTC_path}")
except FileNotFoundError:
    print(f"❌ Error: The file {NTC_path} was not found.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ Successfully loaded NTC_dic from /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/intermediate_data/Europe/DE/2035/NTC_dic.json


In [ ]:
NTC_dic['Electricity'].keys()

dict_keys(['internal', 'import', 'export', 'offshore_direct', 'offshore_offshore', 'virtual', 'ext_ext', 'offshore_hub'])

## 6, Batteries and Electrolysers

In [ ]:
import pandas as pd

battery_data = []

for zone in zones:
    if zone in capacities and 'Battery' in capacities[zone]:
        b_caps = capacities[zone]['Battery']

        # Extract Utility parameters
        utility = b_caps.get('Battery Utility Scale', {})
        # Extract Residential parameters
        residential = b_caps.get('Battery Residential', {})

        battery_data.append({
            'Zone': zone,
            'Utility_P_nom (MW)': utility.get('Net maximum capacity - generation perspective  (MW)', 0.0),
            'Utility_E_nom (MWh)': utility.get('Storage capacity  (MWh)', 0.0),
            'Utility_Efficiency': utility.get('Average efficiency', 0.0),
            'Residential_P_nom (MW)': residential.get('Net maximum capacity - generation perspective  (MW)', 0.0),
            'Residential_E_nom (MWh)': residential.get('Storage capacity  (MWh)', 0.0),
            'Residential_Efficiency': residential.get('Average efficiency', 0.0)
        })

# Create DataFrame
df_batteries = pd.DataFrame(battery_data).set_index('Zone')

# Display the structured data
print("--- Utility vs Residential Battery Parameters across Zones ---")
display(df_batteries)

--- Utility vs Residential Battery Parameters across Zones ---


,Utility_P_nom (MW),Utility_E_nom (MWh),Utility_Efficiency,Residential_P_nom (MW),Residential_E_nom (MWh),Residential_Efficiency
Zone,,,,,,
AL00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
AT00,3600.000000,7200.000000,0.920000,0.000000,0.000000,0.00000
BA00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
BE00,2508.000000,9513.105011,0.920000,1343.000000,2686.000000,0.92000
BG00,1120.000000,2680.000000,0.900000,0.000000,0.000000,0.00000
CH00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
CY00,160.000000,400.000000,0.950000,0.000000,0.000000,0.00000
CZ00,2031.030000,4062.060000,0.920000,0.000000,0.000000,0.00000
DE00,28075.780000,56151.560000,0.920000,11476.800000,27544.319999,0.95000


## 7, RES profiles

In [ ]:
print("Solar parameters in each zone:")
for z, data in capacities.items():
    if 'Solar' in data:
        print(f"\nZone {z}:")
        for param, val in data['Solar'].items():
            if isinstance(val, (int, float)):
                print(f"  - {param}: {val:.2f}")
            else:
                print(f"  - {param}: {val}")
    else:
        print(f"\nZone {z}: No Solar data")

Solar parameters in each zone:

Zone AL00:
  - Installed capacities Thermal Solar (GW):: {'Value': 0.0}
  - Installed capacities Photovoltaic (GW):: {'Value': 1.4}
  - Installed capacities Photovoltaic e-market (GW):: {'Value': 1.4}
  - Installed capacities Photovoltaic dedicated (GW):: {'Value': 0.0}
  - Installed capacities Photovoltaic Shared RES - H2Z1 (GW):: {'Value': 0.0}
  - Installed capacities Photovoltaic Shared RES - H2Z2 (GW):: {'Value': 0.0}
  - Installed capacities Rooftop (GW):: {'Value': 0.0}
  - Installed capacities Solar Thermal with Storage (GW):: {'Value': 0.0}
  - Storage capacities Solar Thermal with Storage (GWh):: {'Value': 0.0}

Zone AT00:
  - Installed capacities Thermal Solar (GW):: {'Value': 0.0}
  - Installed capacities Photovoltaic (GW):: {'Value': 20.49995}
  - Installed capacities Photovoltaic e-market (GW):: {'Value': 20.49995}
  - Installed capacities Photovoltaic dedicated (GW):: {'Value': 0.0}
  - Installed capacities Photovoltaic Shared RES - H2Z1 (

### Hydro - RoR, Pondage, Reservoir and Pump Storage

In [ ]:
# Define keywords for hydro components, splitting PS into Open and Closed
hydro_keywords = ['RoR', 'Pondage', 'Res_', 'PS_Open', 'PS_Closed']

# Identify columns matching the keywords
hydro_columns = [col for col in RES_profiles.columns if any(kw in col for kw in hydro_keywords)]

# Group them by type for better visibility
print("Hydro-related Columns By Category:")
for kw in hydro_keywords:
    matches = [c for c in hydro_columns if kw in c]
    if matches:
        label = kw.replace('_', ' ')
        print(f"{label} related: {matches[:3]} ... (Total: {len(matches)})")

Hydro-related Columns By Category:
RoR related: ['AL00_RoR_MW', 'AT00_RoR_MW', 'BA00_RoR_MW'] ... (Total: 56)
Res  related: ['AL00_Res_Inflow_MW', 'AT00_Res_Inflow_MW', 'BA00_Res_Inflow_MW'] ... (Total: 56)
PS Open related: ['AL00_PS_Open_Inflow_MW', 'AT00_PS_Open_Inflow_MW', 'BA00_PS_Open_Inflow_MW'] ... (Total: 56)
PS Closed related: ['AL00_PS_Closed_Inflow_MW', 'AT00_PS_Closed_Inflow_MW', 'BA00_PS_Closed_Inflow_MW'] ... (Total: 56)


### Solar and Wind Capacities

In [ ]:
import json

# Identify solar parameters from a sample zone
sample_zone = next(iter(capacities))
solar_params = list(capacities[sample_zone].get('Solar', {}).keys())

print(f"Solar parameters found in capacities['{sample_zone}']['Solar']:")
for param in solar_params:
    print(f" - {param}")

Solar parameters found in capacities['AL00']['Solar']:
 - Installed capacities Thermal Solar (GW):
 - Installed capacities Photovoltaic (GW):
 - Installed capacities Photovoltaic e-market (GW):
 - Installed capacities Photovoltaic dedicated (GW):
 - Installed capacities Photovoltaic Shared RES - H2Z1 (GW):
 - Installed capacities Photovoltaic Shared RES - H2Z2 (GW):
 - Installed capacities Rooftop (GW):
 - Installed capacities Solar Thermal with Storage (GW):
 - Storage capacities Solar Thermal with Storage (GWh):


In [ ]:
# Identify wind parameters from a sample zone
sample_zone = next(iter(capacities))
wind_params = list(capacities[sample_zone].get('Wind', {}).keys())

print(f"Wind parameters found in capacities['{sample_zone}']['Wind']:")
for param in wind_params:
    print(f" - {param}")

Wind parameters found in capacities['AL00']['Wind']:
 - Installed capacities Onshore wind Total(GW):
 - Installed capacities Onshore wind e-market (GW):
 - Installed capacities Onshore wind dedicated (GW):
 - Installed capacities Onshore wind Shared RES - H2Z1 (GW):
 - Installed capacities Onshore wind Shared RES - H2Z2 (GW):
 - Installed capacities Offshore wind Total (GW):
 - Installed capacities Offshore wind e-market - hub ready (GW):
 - Installed capacities Offshore wind dedicated - hub ready (GW):
 - Installed capacities Offshore wind Shared RES - H2Z1 - hub ready (GW):
 - Installed capacities Offshore wind Shared RES - H2Z2 - hub ready (GW):
 - Installed capacities Offshore wind e-market - radial (GW):
 - Installed capacities Offshore wind dedicated - radial (GW):
 - Installed capacities Offshore wind Shared RES - H2Z1 - radial (GW):
 - Installed capacities Offshore wind Shared RES - H2Z2 - radial (GW):


In [ ]:
def get_cap(zone, category, key):
    data = capacities.get(zone, {}).get(category, {})
    val = data.get(key, 0.0)
    return val.get('Value', 0.0) if isinstance(val, dict) else val

PV_capacities = {}
Wind_Onshore_capacities = {}
Wind_Offshore_capacities = {}

for z in zones:
    PV_capacities[z] = {
        'Total': get_cap(z, 'Solar', 'Installed capacities Photovoltaic (GW):') * 1e3,
        'El_market': get_cap(z, 'Solar', 'Installed capacities Photovoltaic e-market (GW):') * 1e3,
        'DRES': get_cap(z, 'Solar', 'Installed capacities Photovoltaic dedicated (GW):') * 1e3,
        'SRES_z1': get_cap(z, 'Solar', 'Installed capacities Photovoltaic Shared RES - H2Z1 (GW):') * 1e3,
        'SRES_z2': get_cap(z, 'Solar', 'Installed capacities Photovoltaic Shared RES - H2Z2 (GW):') * 1e3
    }

    Wind_Onshore_capacities[z] = {
        'Total': get_cap(z, 'Wind', 'Installed capacities Onshore wind Total(GW):') * 1e3,
        'El_market': get_cap(z, 'Wind', 'Installed capacities Onshore wind e-market (GW):') * 1e3,
        'DRES': get_cap(z, 'Wind', 'Installed capacities Onshore wind dedicated (GW):') * 1e3,
        'SRES_z1': get_cap(z, 'Wind', 'Installed capacities Onshore wind Shared RES - H2Z1 (GW):') * 1e3,
        'SRES_z2': get_cap(z, 'Wind', 'Installed capacities Onshore wind Shared RES - H2Z2 (GW):') * 1e3
    }

    Wind_Offshore_capacities[z] = {
        'Total': get_cap(z, 'Wind', 'Installed capacities Offshore wind Total (GW):') * 1e3,
        'El_market': (get_cap(z, 'Wind', 'Installed capacities Offshore wind e-market - hub ready (GW):') +
                      get_cap(z, 'Wind', 'Installed capacities Offshore wind e-market - radial (GW):')) * 1e3,
        'DRES': (get_cap(z, 'Wind', 'Installed capacities Offshore wind dedicated - hub ready (GW):') +
                 get_cap(z, 'Wind', 'Installed capacities Offshore wind dedicated - radial (GW):')) * 1e3,
        'SRES_z1': (get_cap(z, 'Wind', 'Installed capacities Offshore wind Shared RES - H2Z1 - hub ready (GW):') +
                    get_cap(z, 'Wind', 'Installed capacities Offshore wind Shared RES - H2Z1 - radial (GW):')) * 1e3,
        'SRES_z2': (get_cap(z, 'Wind', 'Installed capacities Offshore wind Shared RES - H2Z2 - hub ready (GW):') +
                    get_cap(z, 'Wind', 'Installed capacities Offshore wind Shared RES - H2Z2 - radial (GW):')) * 1e3
    }

print(f"✅ Successfully prepared PV_capacities, Wind_Onshore_capacities, and Wind_Offshore_capacities for {len(zones)} zones.")

# Verification for a sample zone
sample_z = zones[0]
print(f"\nSample mappings for {sample_z}:")
print(f"PV Capacities: {PV_capacities[sample_z]}")
print(f"Wind Onshore Capacities: {Wind_Onshore_capacities[sample_z]}")
print(f"Wind Offshore Capacities: {Wind_Offshore_capacities[sample_z]}")

✅ Successfully prepared PV_capacities, Wind_Onshore_capacities, and Wind_Offshore_capacities for 56 zones.

Sample mappings for AL00:
PV Capacities: {'Total': 1400.0, 'El_market': 1400.0, 'DRES': 0.0, 'SRES_z1': 0.0, 'SRES_z2': 0.0}
Wind Onshore Capacities: {'Total': 900.0, 'El_market': 900.0, 'DRES': 0.0, 'SRES_z1': 0.0, 'SRES_z2': 0.0}
Wind Offshore Capacities: {'Total': 0.0, 'El_market': 0.0, 'DRES': 0.0, 'SRES_z1': 0.0, 'SRES_z2': 0.0}


### PV Rooftop - Prosumer Profiles

In [ ]:
'''
import pandas as pd

# Dictionary to store the time-series Rooftop PV profiles for each zone
rooftop_data_collection = {}

for z in zones:
    # Retrieve normalized profile
    roof_prof = RES_profiles.get(f"{z}_Rooftop_PV", 0.0)

    # Retrieve capacity and convert GW to MW
    roof_cap = get_cap(z, 'Solar', 'Installed capacities Rooftop (GW):') * 1e3

    # Calculate absolute profile
    rooftop_data_collection[f'{z}_Rooftop_PV'] = roof_cap * roof_prof

# Create DataFrame
Rooftop_df = pd.DataFrame(rooftop_data_collection)

print(f"Successfully created Rooftop_df with shape: {Rooftop_df.shape}")
display(Rooftop_df.head())
'''
print("Cell frozen: Rooftop PV is now handled directly during model_data mapping.")

Cell frozen: Rooftop PV is now handled directly during model_data mapping.


## 8, Hydro metadata

In [ ]:
import os
import json

# Define the path to hydro_metadata.json
hydro_metadata_path = os.path.join(INTER_DIR, 'hydro_metadata.json')

# Load and display the hydro metadata
try:
    with open(hydro_metadata_path, 'r') as f:
        hydro_metadata = json.load(f)
    print(f"✅ Successfully loaded hydro metadata from: {os.path.basename(hydro_metadata_path)}")

    # Display the structure or a sample of the metadata
    import pandas as pd
    # Convert to DataFrame if it's a list of dicts, otherwise display keys
    if isinstance(hydro_metadata, list):
        display(pd.DataFrame(hydro_metadata).head())
    else:
        print("Top-level keys:", list(hydro_metadata.keys()))
        # Display a sample of the first key's content
        first_key = next(iter(hydro_metadata))
        print(f"\nSample data for {first_key}:")
        print(json.dumps(hydro_metadata[first_key], indent=2))

except FileNotFoundError:
    print(f"❌ Error: The file {os.path.basename(hydro_metadata_path)} was not found in {os.path.dirname(hydro_metadata_path)}.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ Successfully loaded hydro metadata from: hydro_metadata.json
Top-level keys: ['AL00', 'AT00', 'BA00', 'BE00', 'BG00', 'CH00', 'CY00', 'CZ00', 'DE00', 'DKE1', 'DKW1', 'EE00', 'ES00', 'FI00', 'FR00', 'GR00', 'GR03', 'HR00', 'HU00', 'IE00', 'ITCA', 'ITCN', 'ITCS', 'ITN1', 'ITS1', 'ITSA', 'ITSI', 'LT00', 'LUF1', 'LUG1', 'LUV1', 'LV00', 'MD00', 'ME00', 'MK00', 'MT00', 'NL00', 'NOM1', 'NON1', 'NOS1', 'NOS2', 'NOS3', 'PL00', 'PT00', 'RO00', 'RS00', 'SE01', 'SE02', 'SE03', 'SE04', 'SI00', 'SK00', 'TR00', 'UA00', 'UK00', 'UKNI']

Sample data for AL00:
{
  "Filename": "PEMMDB_AL00_NationalTrends_2035.xlsx",
  "Market Node:": "AL00",
  "Reference year": 2035,
  "Run of River - MW": 534.762,
  "Pondage - GWh": 0,
  "Pondage - MW": 0,
  "Reservoir - GWh": 1721.68,
  "Reservoir - MW": 2031.744,
  "PS Open - GWh": 0,
  "PS Open (turbine) - MW": 0,
  "PS Open (pump) - MW": 0,
  "PS Closed - GWh": 0,
  "PS Closed (turbine) - MW": 0,
  "PS Closed (pump) - MW": 0
}


## 9, Gas Infrastructure Data

In [ ]:
GAS_INFRA_DIR_2024 = os.path.join(PROJECT_DIR, "TYNDP_scenario_2024", input_data["inter_dir"], input_data["project_name"], input_data["scenario"], str(input_data["year"]))

gas_infra_path = os.path.join(GAS_INFRA_DIR_2024, "gas_infrastructure.json")

try:
    with open(gas_infra_path, 'r') as f:
        gas_infra_dic = json.load(f)

    LNG = pd.DataFrame(gas_infra_dic['LNG'])
    Storage = pd.DataFrame(gas_infra_dic['Storage'])
    NTC_gas_df = pd.DataFrame(gas_infra_dic['Cross_border']['Internal'])
    Import_NTC_gas = pd.DataFrame(gas_infra_dic['Cross_border']['Import'])
    Export_NTC_gas = pd.DataFrame(gas_infra_dic['Cross_border']['Export'])

    print("✅ 2024 Gas Infrastructure Capacities are imported: LNG, Storage, NTC_gas_df, Import_NTC_gas, and Export_NTC_gas.")

except FileNotFoundError:
    print(f"❌ Error: The file {os.path.basename(gas_infra_path)} was not found in {os.path.dirname(gas_infra_path)}.")
except Exception as e:
    print(f"❌ An error occurred: {e}")

❌ Error: The file gas_infrastructure.json was not found in /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2024/intermediate_data/Europe/DE/2035.


## 10, Prices

In [ ]:
PRICES_DIR = os.path.join(PROJECT_DIR,str(input_data["data_set"]),input_data['inter_dir'])

In [ ]:
import os
import pandas as pd

prices_file_path = os.path.join(PRICES_DIR, 'prices.csv')

try:
    # Read the CSV and set 'Fuel' as the index
    prices_df = pd.read_csv(prices_file_path)
    if 'Fuel' in prices_df.columns:
        prices_df.set_index('Fuel', inplace=True)
    display(prices_df.head())
except FileNotFoundError:
    print(f"❌ Error: The file {os.path.basename(prices_file_path)} was not found.")
    print(f"Files in {PRICES_DIR}:")
    if os.path.exists(PRICES_DIR):
        for item in os.listdir(PRICES_DIR):
            print(f"  - {item}")
    else:
        print("  Directory does not exist.")

,2030,2035,2040,2050
Fuel,,,,
Nuclear,2.187882,2.187882,2.187882,2.187882
Lignite G1 (BG - MK - CZ),6.692417,6.692417,6.692417,6.692417
Lignite G2 (SK - DE - RS - PL - ME - UKNI - BA - IE),8.604536,8.604536,8.604536,8.604536
Lignite G3 (SL - RO - HU),11.329306,11.329306,11.329306,11.329306
Lignite G4 (GR - TR),14.818923,14.818923,14.818923,14.818923


In [ ]:
import pandas as pd

def get_price_idx(keyword):
    # Helper to find the exact index matching the keyword
    match = [idx for idx in prices_df.index if keyword in str(idx)]
    return match[0] if match else None

# Adding commodities prices as input in model data
# Mapping values dynamically from prices_df for the current year
prices = {
      "Hard coal (EUR/t)" : prices_df.loc['Hard coal', str(year)],
      "CO2 (EUR/tCO2)" : prices_df.loc['CO2 price', str(year)],
      "Crude oil (EUR/MWh)" : prices_df.loc['Crude oil', str(year)],
      "Light oil (EUR/MWh)" : prices_df.loc['Light oil', str(year)],
      "Heavy oil (EUR/MWh)" : prices_df.loc['Heavy oil', str(year)],
      "Synthetic Methane (EUR/MWh)" : prices_df.loc[get_price_idx('Synthetic Methane'), str(year)],
      "Biomethane (EUR/MWh)" : prices_df.loc[get_price_idx('Biomethane'), str(year)],
      "H2 (EUR/MWh)" : prices_df.loc['Hydrogen', str(year)],
      "Lignite G1" : prices_df.loc[get_price_idx('Lignite G1'), str(year)],
      "Lignite G2" : prices_df.loc[get_price_idx('Lignite G2'), str(year)],
      "Lignite G3" : prices_df.loc[get_price_idx('Lignite G3'), str(year)],
      "Lignite G4" : prices_df.loc[get_price_idx('Lignite G4'), str(year)],
      "Oil Shale (EUR/MWh)" : prices_df.loc['Oil Shale', str(year)],
      "Natural Gas (EUR/MWh)" : prices_df.loc['Natural Gas', str(year)]
}

#"Ammonia (EUR/MWh)" : prices_df.loc[get_price_idx('Amonia'), str(year)],


## 11, Common Data

In [ ]:
import os
import json

# Construct the path specifically for scenario 2026 and intermediate data
common_data_path = os.path.join(PROJECT_DIR, "TYNDP_scenario_2026", "intermediate_data", "CommonData.json")

try:
# Load the JSON file into a dictionary
  with open(common_data_path, 'r') as f:
    CommonData_dic = json.load(f)
    print(f"✅ Successfully loaded CommonData_dic from: {common_data_path}")
    # Display the first 5 top-level keys as confirmation
    print("Top-level keys found:", list(CommonData_dic.keys())[:5])
except FileNotFoundError:
    print(f"❌ Error: The file CommonData.json was not found at {common_data_path}")
except Exception as e:
    print(f"❌ An error occurred while loading the JSON: {e}")

✅ Successfully loaded CommonData_dic from: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/intermediate_data/CommonData.json
Top-level keys found: ['Nuclear, -', 'Hard coal, old 1', 'Hard coal, old 2', 'Hard coal, new', 'Hard coal, CCS']


In [ ]:
CommonData_dic['Nuclear, -']

{'Efficiency range in NCV terms, %': '30% - 35%',
 'Standard efficiency in NCV terms, %': 0.33,
 'CO2 emission factor, kg / Net GJ': 0,
 'Variable O&M cost, €/MWh': 9,
 'Min Time on, hours': 12,
 'Min Time off, hours': 12,
 'Start-up fuel consumption - warm start, Net GJ /MW. start': 14,
 'Start-up fix cost (e.g. wear) warm start, € /MW. start': 21,
 'Start-up fuel consumption - cold start, Net GJ /MW. start': None,
 'Start-up fix cost (e.g. wear) cold start, € /MW. start': None,
 'Start-up fuel consumption - hot start, Net GJ /MW. start': None,
 'Start-up fix cost (e.g. wear) hot start, € /MW. start': None,
 'Transition time [h] from hot to warm': None,
 'Transition time [h] from hot to cold': None,
 'Unavailability, Forced outage, annual rate, %': 0.05,
 'Mean time to repair, Days': 7,
 'Planned outage, annual rate, number of days': 54,
 'winter, % of annual number of days': 0.15,
 'Minimum stable generation, (% of max power)': 0.4,
 'Ramp up rate, % of max output power / min': 0.05,

# Model Data

In [ ]:
model_data = {
    "Europe": {},
    "Regional": {},
    "Zonal": {},
    "Local": {}
}

# Initialize zonal sub-dictionaries for each project zone
for z in zones:
    model_data["Zonal"][z] = {}

# Add sublayers to Regional level
model_data["Regional"]["electricity"] = {}
model_data["Regional"]["H2"] = {}
model_data["Regional"]["CH4"] = {}


## Europe level

In [ ]:
# Map to the Global segment of the hierarchical model
model_data["Europe"]["prices"] = prices

print("✅ Added commodity prices to model_data['Europe']['prices']")

✅ Added commodity prices to model_data['Europe']['prices']


In [ ]:
model_data['Europe']['CommonData'] = CommonData_dic

print("✅ Successfully added CommonData_dic to model_data['CommonData']")

✅ Successfully added CommonData_dic to model_data['CommonData']


In [ ]:
# Initialize eFuel key in the Global level of model_data
if "eFuel" not in model_data["Europe"]:
    model_data["Europe"]["eFuel"] = {}

# Add the profiles from Global_efuels_df into model_data
if 'Global_efuels_df' in locals():
    model_data["Europe"]["eFuel"]["SNG"] = Global_efuels_df["SNG"]
    model_data["Europe"]["eFuel"]["eDiesel"] = Global_efuels_df["eDiesel"]
    model_data["Europe"]["eFuel"]["eKerosine"] = Global_efuels_df["eKerosine"]

    print("✅ Successfully incorporated Global eFuel profiles into model_data['Global']['eFuel']:")
    print(list(model_data["Europe"]["eFuel"].keys()))
else:
    print("❌ Error: Global_efuels_df is not defined in the current namespace.")

✅ Successfully incorporated Global eFuel profiles into model_data['Global']['eFuel']:
['SNG', 'eDiesel', 'eKerosine']


## Regional Level

### NTCs

In [ ]:
NTC_dic['Hydrogen']['ammonia']

[{'Border': 'Ammonia_BE-BEh2',
  'zone 1': 'Ammonia_BE',
  'zone 2': 'BEh2',
  'capacity, MW': 3788.8417916666663,
  'losses': 1.0},
 {'Border': 'Ammonia_DE-DEh2',
  'zone 1': 'Ammonia_DE',
  'zone 2': 'DEh2',
  'capacity, MW': 2390.6638333333335,
  'losses': 1.0},
 {'Border': 'Ammonia_FRSW-FRh2SW',
  'zone 1': 'Ammonia_FRSW',
  'zone 2': 'FRh2SW',
  'capacity, MW': 0.0,
  'losses': 1.0},
 {'Border': 'Ammonia_FRW-FRh2',
  'zone 1': 'Ammonia_FRW',
  'zone 2': 'FRh2',
  'capacity, MW': 0.0,
  'losses': 1.0},
 {'Border': 'Ammonia_FRN-FRh2N',
  'zone 1': 'Ammonia_FRN',
  'zone 2': 'FRh2N',
  'capacity, MW': 1694.91525,
  'losses': 1.0},
 {'Border': 'Ammonia_IT-IB_ITh2',
  'zone 1': 'Ammonia_IT',
  'zone 2': 'IB_ITh2',
  'capacity, MW': 0.0,
  'losses': 1.0},
 {'Border': 'Ammonia_UK-UKh2',
  'zone 1': 'Ammonia_UK',
  'zone 2': 'UKh2',
  'capacity, MW': 0.0,
  'losses': 1.0},
 {'Border': 'Ammonia_NL-NLh2',
  'zone 1': 'Ammonia_NL',
  'zone 2': 'NLh2',
  'capacity, MW': 4812.853107344633,
  '

In [ ]:
# Map regional-level data to the model_data structure
import pandas as pd

# 1. Electricity NTCs
model_data["Regional"]["electricity"]["NTC"] = NTC_dic['Electricity']

# 2. Hydrogen Data
model_data["Regional"]["H2"]["NTC"] = NTC_dic['Hydrogen']

# 3. Natural Gas (CH4)
model_data["Regional"]["CH4"]["LNG"] = LNG if 'LNG' in locals() else pd.DataFrame()
model_data["Regional"]["CH4"]["Storage"] = Storage if 'Storage' in locals() else pd.DataFrame()
model_data["Regional"]["CH4"]["NTC"] = {
    "Internal": NTC_gas_df if 'NTC_gas_df' in locals() else pd.DataFrame(),
    "Import": Import_NTC_gas if 'Import_NTC_gas' in locals() else pd.DataFrame(),
    "Export": Export_NTC_gas if 'Export_NTC_gas' in locals() else pd.DataFrame()
}

print("✅ model_data['Regional'] successfully populated.")

✅ model_data['Regional'] successfully populated.


### Offshore - capacities and profiles

In [ ]:
# Ensure the Offshore dictionary exists
if 'Offshore' not in model_data['Regional']['electricity']:
    model_data['Regional']['electricity']['Offshore'] = {}

# Dynamically find disconnected offshore regions based on NTCs
ntc_data = model_data['Regional']['electricity'].get('NTC', {})
connected_zones = set()
for cat, links in ntc_data.items():
    if isinstance(links, list):
        for link in links:
            z1 = link.get('zone 1', '')
            z2 = link.get('zone 2', '')
            if isinstance(z1, str) and z1.endswith('_OFF'):
                connected_zones.add(z1)
            if isinstance(z2, str) and z2.endswith('_OFF'):
                connected_zones.add(z2)

all_offshore_regions = set()
if 'offshore_capacities' in locals() and 'wind' in offshore_capacities:
    all_offshore_regions.update(offshore_capacities['wind'].keys())
if 'Offshore_profiles_df' in locals():
    all_offshore_regions.update(Offshore_profiles_df.columns)

disconnected_regions = list(all_offshore_regions - connected_zones)
print(f"Dynamically identified disconnected offshore regions: {sorted(disconnected_regions)}\n")

# 1. Incorporate offshore capacities from the 'wind' sub-category
if 'offshore_capacities' in locals() and 'wind' in offshore_capacities:
    # Remove disconnected regions from the 'wind' capacities
    clean_offshore_capacities = {k: v for k, v in offshore_capacities['wind'].items() if k not in disconnected_regions}

    # Sync with profiles to ensure we have the exact same offshore_names
    if 'Offshore_profiles_df' in locals():
        for region in Offshore_profiles_df.columns:
            if region not in clean_offshore_capacities and region not in disconnected_regions:
                clean_offshore_capacities[region] = {} # Assign empty dict for connected regions missing capacities

    model_data['Regional']['electricity']['Offshore']['capacities'] = clean_offshore_capacities
    print("✅ Successfully incorporated offshore wind capacities (excluding disconnected regions) into model_data.")
else:
    print("❌ Error: offshore_capacities['wind'] is not defined.")

# 2. Incorporate offshore profiles
if 'Offshore_profiles_df' in locals():
    clean_profiles = Offshore_profiles_df.copy()

    # Ensure none of the disconnected regions are in the profiles either
    cols_to_drop = [c for c in disconnected_regions if c in clean_profiles.columns]
    if cols_to_drop:
        clean_profiles = clean_profiles.drop(columns=cols_to_drop)

    model_data['Regional']['electricity']['Offshore']['profiles'] = clean_profiles
    print("✅ Successfully incorporated offshore profiles into model_data.")
else:
    print("❌ Error: Offshore_profiles_df is not defined.")

# Verify the updated structure
print("\n--- Regional Offshore Layer Keys ---")
print(list(model_data['Regional']['electricity']['Offshore'].keys()))
if 'clean_offshore_capacities' in locals() and 'clean_profiles' in locals():
    print(f"Total synced offshore regions -> Capacities: {len(clean_offshore_capacities)}, Profiles: {len(clean_profiles.columns)}")

Dynamically identified disconnected offshore regions: ['DKBF_OFF', 'DKN6_OFF', 'DKN7_OFF', 'DKN8_OFF', 'DKN9_OFF', 'ESA1_OFF', 'ESA2_OFF', 'ESAS_OFF', 'ESG1_OFF', 'NL0R_OFF']

✅ Successfully incorporated offshore wind capacities (excluding disconnected regions) into model_data.
✅ Successfully incorporated offshore profiles into model_data.

--- Regional Offshore Layer Keys ---
['capacities', 'profiles']
Total synced offshore regions -> Capacities: 44, Profiles: 44


### H2 imports

In [ ]:
import pandas as pd

# Filter H2_data['Import_data'] based on Fuel type
pure_h2_data = [item for item in H2_data['Import_data'] if item.get('Fuel') == 'Pure Hydrogen']
ammonia_data = [item for item in H2_data['Import_data'] if item.get('Fuel') == 'Ammonia']

# Create the DataFrames with the renamed variable df_pipeline
df_pipeline = pd.DataFrame(pure_h2_data)
df_ammonia = pd.DataFrame(ammonia_data)

In [ ]:
# Reset and initialize the add_Import dictionary within the H2 Regional layer
model_data['Regional']['H2']['add_Import'] = {}

# Create the nested structure for pipeline, ammonia, and profiles
model_data['Regional']['H2']['add_Import']['pipeline'] = df_pipeline
model_data['Regional']['H2']['add_Import']['ammonia'] = df_ammonia
model_data['Regional']['H2']['add_Import']['profile'] = H2_import_profiles

print("✅ Successfully structured model_data['Regional']['H2']['add_Import'] with 'pipeline', 'ammonia', and 'profile'.")

# Verification of the updated keys
print("\n--- Keys in model_data['Regional']['H2']['add_Import'] ---")
print(list(model_data['Regional']['H2']['add_Import'].keys()))

✅ Successfully structured model_data['Regional']['H2']['add_Import'] with 'pipeline', 'ammonia', and 'profile'.

--- Keys in model_data['Regional']['H2']['add_Import'] ---
['pipeline', 'ammonia', 'profile']


### H2 Bottlenecks

In [ ]:
model_data['Regional']['H2']['Bottlenecks'] = ['IB_ITh2','IB_SKh2C','IB_SKh2E','PLh2nbc','UKh2/INT','IB_GRh2P']

### Offshore Electrolysers

In [ ]:
model_data['Regional']['H2']['Electrolysers'] = offshore_capacities['electrolyser']

## Zonal Level

### Demand Profiles

#### El_Market and Prosumer Demands and EVs

In [ ]:
for z in zones:
    # Ensure the zonal and Demand containers exist
    if z not in model_data['Zonal']:
        model_data['Zonal'][z] = {}
    if 'Demand' not in model_data['Zonal'][z]:
        model_data['Zonal'][z]['Demand'] = {}

    # Identify electricity demand columns for this zone (Market and Prosumer)
    elec_cols = [col for col in demand_profiles.columns if col.startswith(f"{z}_") and ('El_market' in col or 'El_prosumer' in col)]

    for col in elec_cols:
        # Strip the zone prefix to create a clean key (e.g., 'El_market')
        profile_key = col.replace(f"{z}_", "", 1)
        model_data['Zonal'][z]['Demand'][profile_key] = demand_profiles[col]

print(f"✅ Successfully mapped electricity demand profiles across {len(zones)} zones in model_data['Zonal'][z]['Demand'].")

# Verification for a sample zone
example_z = zones[0]
print(f"\nAvailable demand keys for {example_z} Demand: {list(model_data['Zonal'][example_z]['Demand'].keys())}")

✅ Successfully mapped electricity demand profiles across 56 zones in model_data['Zonal'][z]['Demand'].

Available demand keys for AL00 Demand: ['El_market', 'El_prosumer', 'EV_El_market', 'EV_El_prosumer']


#### Heat Demand - CH4

In [ ]:
import pandas as pd

# Dictionary to track CH4 mapping progress
ch4_heat_mapping_count = 0

for z in zones:
    # Define the specific column name for CH4 heat in this zone
    ch4_col = f"{z}_CH4_heat"

    if ch4_col in demand_profiles.columns:
        # Ensure the zonal and Demand containers exist
        if z not in model_data['Zonal']:
            model_data['Zonal'][z] = {}
        if 'Demand' not in model_data['Zonal'][z]:
            model_data['Zonal'][z]['Demand'] = {}

        # Map the profile specifically under 'CH4_heat'
        model_data['Zonal'][z]['Demand']['CH4_heat'] = demand_profiles[ch4_col]
        ch4_heat_mapping_count += 1
    else:
        # Initialize with 0.0 if not found to maintain structural consistency
        if z in model_data['Zonal'] and 'Demand' in model_data['Zonal'][z]:
             model_data['Zonal'][z]['Demand']['CH4_heat'] = 0.0

print(f"✅ Successfully mapped CH4_heat profiles for {ch4_heat_mapping_count} zones into model_data['Zonal'].")

# Verification for a sample zone
sample_z = 'DE00'
if 'CH4_heat' in model_data['Zonal'].get(sample_z, {}).get('Demand', {}):
    print(f"Sample Check ({sample_z}): 'CH4_heat' key is present and populated.")

✅ Successfully mapped CH4_heat profiles for 56 zones into model_data['Zonal'].
Sample Check (DE00): 'CH4_heat' key is present and populated.


#### H2 Zones Mapping and demands

In [ ]:
H2_zones = [
    'ALh2', 'ATh2', 'BAh2', 'BEh2', 'BGh2', 'CHh2', 'CYh2', 'CZh2', 'DEh2',
    'DEh2Z1', 'DKh2', 'EEh2', 'ESh2', 'FIh2', 'FIh2Al', 'FIh2N', 'FIh2S',
    'FRh2', 'FRh2N', 'FRh2S', 'FRh2SW', 'GRh2', 'HRh2', 'HUh2', 'IEh2',
    'ITh2', 'LTh2', 'LTh2Z1', 'LUh2', 'LVh2', 'MDh2', 'MKh2', 'MTh2',
    'NLh2', 'NOh2', 'PLh2', 'PTh2', 'PTh2Z1', 'ROh2', 'RSh2', 'SEh2',
    'SIh2', 'SKh2E', 'SKh2W', 'UKh2'
]

In [ ]:
zone_mapping = {}
assigned_h2_zones = set()

# Define the sequence to prioritize ITN1 for ITh2
for zone in zones:
    prefix = zone[:2]
    matched_h2 = [h2 for h2 in H2_zones if h2.startswith(prefix)]

    # Special logic: If this is an Italian zone but NOT ITN1, don't let it take ITh2 yet
    if prefix == 'IT' and zone != 'ITN1':
        matched_h2 = [h2 for h2 in matched_h2 if h2 != 'ITh2']

    # Manual injection for ITN1 if it somehow wasn't in the prefix match
    if zone == 'ITN1' and 'ITh2' not in assigned_h2_zones:
        if 'ITh2' not in matched_h2: matched_h2.append('ITh2')

    mapped_h2 = []
    for h2 in matched_h2:
        if h2 not in assigned_h2_zones:
            mapped_h2.append(h2)
            assigned_h2_zones.add(h2)

    zone_mapping[zone] = mapped_h2

In [ ]:
# Consolidated Mapping of Hydrogen Demand Profiles
# Ensures all zones have 'H2_demand' key for structural consistency

for ez, h2_profiles in zone_mapping.items():
    # Ensure the zonal container exists
    if ez not in model_data['Zonal']:
        model_data['Zonal'][ez] = {}

    # Initialize/Reset H2_demand as an empty dictionary for every zone (uniform API)
    model_data['Zonal'][ez]['H2_demand'] = {}

    # Populate profiles if assigned in the mapping and present in demand_profiles
    if h2_profiles:
        for h2_col in h2_profiles:
            if h2_col in demand_profiles.columns:
                # Store the profile series under its original column name
                model_data['Zonal'][ez]['H2_demand'][h2_col] = demand_profiles[h2_col]

# Verification of the mapping
total_h2_mapped = sum(len(model_data['Zonal'][z].get('H2_demand', {})) for z in zones)
empty_h2_zones = [z for z in zones if not model_data['Zonal'][z]['H2_demand']]

print(f"✅ Successfully mapped {total_h2_mapped} H2 profiles into model_data['Zonal'].")
print(f"Zones initialized with empty H2_demand dictionaries: {len(empty_h2_zones)}")
if empty_h2_zones:
    print(f"Sample empty zones: {empty_h2_zones[:5]}...")

✅ Successfully mapped 45 H2 profiles into model_data['Zonal'].
Zones initialized with empty H2_demand dictionaries: 21
Sample empty zones: ['DKW1', 'GR03', 'ITCA', 'ITCN', 'ITCS']...


#### Heat Demand - H2

In [ ]:
demand_profiles

,ALh2,ATh2,BAh2,BEh2,BGh2,CHh2,CYh2,CZh2,DEh2,DEh2Z1,...,UK00_CH4_heat,UK00_El_market,UK00_El_prosumer,UK00_EV_El_market,UK00_EV_El_prosumer,UKNI_CH4_heat,UKNI_El_market,UKNI_El_prosumer,UKNI_EV_El_market,UKNI_EV_El_prosumer
2035-01-01 00:00:00,0.0,1784.593822,0.0,2552.157013,555.167098,118.54379,0.367639,1817.387207,14986.606035,2043.628096,...,0.0,40238.819227,0.0,0.0,0.0,242.598920,995.245808,0.0,0.0,0.0
2035-01-01 01:00:00,0.0,1784.593822,0.0,2552.157013,555.156343,118.54379,0.367639,1817.387207,14997.243012,2045.078593,...,0.0,39336.924482,0.0,0.0,0.0,242.605863,982.711240,0.0,0.0,0.0
2035-01-01 02:00:00,0.0,1784.593822,0.0,2552.157013,555.148437,118.54379,0.367639,1817.387207,15029.138206,2049.427937,...,0.0,37441.256854,0.0,0.0,0.0,242.706138,946.471496,0.0,0.0,0.0
2035-01-01 03:00:00,0.0,1784.593822,0.0,2552.157013,555.148437,118.54379,0.367639,1817.387207,15056.185984,2053.116271,...,0.0,35734.752744,0.0,0.0,0.0,243.587023,924.870418,0.0,0.0,0.0
2035-01-01 04:00:00,0.0,1787.546245,0.0,2552.157013,555.450504,118.54379,0.367639,1818.188387,15208.542332,2073.892136,...,0.0,34227.446651,0.0,0.0,0.0,248.430346,925.785844,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2035-12-31 19:00:00,0.0,1636.775932,0.0,2552.157013,555.887121,118.54379,0.367639,1313.872731,15426.912122,2103.669835,...,0.0,64493.383041,0.0,0.0,0.0,362.683036,1697.974642,0.0,0.0,0.0
2035-12-31 20:00:00,0.0,1633.823508,0.0,2552.157013,555.625832,118.54379,0.367639,1313.071551,15048.401368,2052.054732,...,0.0,60657.926517,0.0,0.0,0.0,339.911937,1593.000059,0.0,0.0,0.0
2035-12-31 21:00:00,0.0,1630.871085,0.0,2552.157013,555.239182,118.54379,0.367639,1312.270371,14636.692618,1995.912630,...,0.0,55853.823222,0.0,0.0,0.0,312.718674,1482.843236,0.0,0.0,0.0
2035-12-31 22:00:00,0.0,1627.918661,0.0,2552.157013,554.800926,118.54379,0.367639,1311.469192,14341.299695,1955.631777,...,0.0,50508.048517,0.0,0.0,0.0,295.162690,1355.583034,0.0,0.0,0.0


In [ ]:
import pandas as pd

# Dictionary to track H2 heat mapping progress
h2_heat_mapping_count = 0

for ez, h2_zones in zone_mapping.items():
    # Ensure the zonal and Demand containers exist
    if ez not in model_data['Zonal']:
        model_data['Zonal'][ez] = {}
    if 'Demand' not in model_data['Zonal'][ez]:
        model_data['Zonal'][ez]['Demand'] = {}

    # Initialize/Reset H2_heat as an empty dictionary for every zone
    model_data['Zonal'][ez]['Demand']['H2_heat'] = {}

    # Map profiles if assigned in the mapping and present in demand_profiles
    if h2_zones:
        for h2_zone in h2_zones:
            # Construct the heat column name (e.g., 'DEh2_H2_heat')
            h2_heat_col = f"{h2_zone}_H2_heat"

            if h2_heat_col in demand_profiles.columns:
                # Store the profile series
                model_data['Zonal'][ez]['Demand']['H2_heat'][h2_zone] = demand_profiles[h2_heat_col]
                h2_heat_mapping_count += 1

print(f"✅ Successfully mapped {h2_heat_mapping_count} H2_heat profiles across electricity zones in model_data['Zonal'].")

# Verification for a sample zone
sample_ez = 'DE00'
if 'H2_heat' in model_data['Zonal'].get(sample_ez, {}).get('Demand', {}):
    found_keys = list(model_data['Zonal'][sample_ez]['Demand']['H2_heat'].keys())
    print(f"Sample Check ({sample_ez}): Found H2 heat profiles for: {found_keys}")

✅ Successfully mapped 45 H2_heat profiles across electricity zones in model_data['Zonal'].
Sample Check (DE00): Found H2 heat profiles for: ['DEh2', 'DEh2Z1']


### Hydro

#### Profiles - RoR, Reservoir, PS-Closed and PS-Open

In [ ]:
import pandas as pd

# Define the hydro profiles to be added to model_data['Zonal']
hydro_profile_keys = {
    'RoR_MW': 'RoR_MW',
    'Res_Inflow_MW': 'RES_Inflow_MW',
    'PS_Open_Inflow_MW': 'PS_Open_Inflow_MW',
    'PS_Closed_Inflow_MW': 'PS_Closed_Inflow_MW'
}

for z in zones:
    # Ensure the zonal container exists
    if z not in model_data['Zonal']:
        model_data['Zonal'][z] = {}

    zonal_tier = model_data['Zonal'][z]

    # Ensure hydro dict exists
    if 'hydro' not in zonal_tier:
        zonal_tier['hydro'] = {'profiles': {}, 'capacities': {}}

    for raw_key, model_key in hydro_profile_keys.items():
        col_name = f"{z}_{raw_key}"
        if col_name in RES_profiles.columns:
            zonal_tier['hydro']['profiles'][model_key] = RES_profiles[col_name]
        else:
            # If profile is missing, initialize with 0 for consistency
            zonal_tier['hydro']['profiles'][model_key] = 0.0

print(f"✅ Successfully added {list(hydro_profile_keys.values())} to all zones in model_data['Zonal'][zone]['hydro']['profiles'].")

# Verification
sample_z = zones[0]
print(f"\nUpdated hydro profiles in model_data['Zonal']['{sample_z}']['hydro']['profiles']:")
for key in hydro_profile_keys.values():
    status = "Present" if key in model_data['Zonal'][sample_z]['hydro']['profiles'] else "Missing"
    print(f" - {key}: {status}")


✅ Successfully added ['RoR_MW', 'RES_Inflow_MW', 'PS_Open_Inflow_MW', 'PS_Closed_Inflow_MW'] to all zones in model_data['Zonal'][zone]['hydro']['profiles'].

Updated hydro profiles in model_data['Zonal']['AL00']['hydro']['profiles']:
 - RoR_MW: Present
 - RES_Inflow_MW: Present
 - PS_Open_Inflow_MW: Present
 - PS_Closed_Inflow_MW: Present


#### Metadata Parameters

In [ ]:
import pandas as pd

# List of all target metrics to extract from hydro_metadata
hydro_target_metrics = [
    "Run of River - MW",
    "Pondage - GWh",
    "Pondage - MW",
    "Reservoir - GWh",
    "Reservoir - MW",
    "PS Open - GWh",
    "PS Open (turbine) - MW",
    "PS Open (pump) - MW",
    "PS Closed - GWh",
    "PS Closed (turbine) - MW",
    "PS Closed (pump) - MW"
]

# Integrate all Hydro metadata parameters into model_data['Zonal']
for zone in zones:
    # Ensure the zonal container exists
    if zone not in model_data['Zonal']:
        model_data['Zonal'][zone] = {}

    zonal_tier = model_data['Zonal'][zone]

    if 'hydro' not in zonal_tier:
        zonal_tier['hydro'] = {'profiles': {}, 'capacities': {}}

    # Retrieve metadata for the zone if it exists
    meta = hydro_metadata.get(zone, {})

    for metric in hydro_target_metrics:
        # Clean the key name for internal dictionary storage
        clean_key = metric.replace(" - ", "_").replace(" (", "_").replace(")", "").replace(" ", "_")

        # Map the value, ensuring it is a float, defaulting to 0.0 if not found
        val = meta.get(metric, 0.0)
        try:
            zonal_tier['hydro']['capacities'][clean_key] = float(val)
        except (ValueError, TypeError):
            zonal_tier['hydro']['capacities'][clean_key] = 0.0

print(f"✅ Successfully incorporated all 11 Hydro metadata parameters as floats into model_data['Zonal'][zone]['hydro']['capacities'] for {len(zones)} zones.")

# Verification output for a sample zone
sample_z = zones[0]
print(f"\nHydro parameters (Float check) for {sample_z}:")
for metric in hydro_target_metrics:
    ck = metric.replace(" - ", "_").replace(" (", "_").replace(")", "").replace(" ", "_")
    val = model_data['Zonal'][sample_z]['hydro']['capacities'].get(ck)
    print(f" - {ck}: {val} (Type: {type(val).__name__})")


✅ Successfully incorporated all 11 Hydro metadata parameters as floats into model_data['Zonal'][zone]['hydro']['capacities'] for 56 zones.

Hydro parameters (Float check) for AL00:
 - Run_of_River_MW: 534.762 (Type: float)
 - Pondage_GWh: 0.0 (Type: float)
 - Pondage_MW: 0.0 (Type: float)
 - Reservoir_GWh: 1721.68 (Type: float)
 - Reservoir_MW: 2031.744 (Type: float)
 - PS_Open_GWh: 0.0 (Type: float)
 - PS_Open_turbine_MW: 0.0 (Type: float)
 - PS_Open_pump_MW: 0.0 (Type: float)
 - PS_Closed_GWh: 0.0 (Type: float)
 - PS_Closed_turbine_MW: 0.0 (Type: float)
 - PS_Closed_pump_MW: 0.0 (Type: float)


### Electrolysers

In [ ]:
capacities['ITCA']['Electrolyser']['Electrolyser e-market Z2'].keys()

dict_keys(['Net maximum capacity  (MW)', 'Number of units', 'Average efficiency', 'H2 storage  (GWh)', 'Ramp up rate  (MW/h)', 'Ramp down rate  (MW/h)', 'Fixed generation reduction (% of max power output)'])

In [ ]:
# List of electrolyser types to extract into model_data
electrolyser_types = [
    'Electrolyser e-market Z2',
    'Electrolyser Shared RES Z2',
    'Electrolyser Dedicated RES Z2',
    'Electrolyser e-market Z1',
    'Electrolyser Shared RES Z1',
    'Electrolyser Dedicated RES Z1'
]

In [ ]:
import pandas as pd

# 1. Configuration for Mapping
# Mapping keys from the capacity.json structure to model_data naming convention
key_map = {
    'Net maximum capacity  (MW)': 'P_nom',
    'Number of units': 'number_of_units',
    'Average efficiency': 'efficiency',
    'H2 storage  (GWh)': 'H2_storage_GWh',
    'Ramp up rate  (MW/h)': 'ramp_up',
    'Ramp down rate  (MW/h)': 'ramp_down',
    'Fixed generation reduction (% of max power output)': 'fixed_generation_reduction_pct'
}

# Mapping and naming for the sub-dictionaries
type_mapping = {
    'Electrolyser e-market Z2': 'el_market',
    'Electrolyser Shared RES Z2': 'SRES',
    'Electrolyser Dedicated RES Z2': 'DRES'
}

# 2. Execution of Zonal Restructuring
for z in zones:
    # Ensure the zonal container exists
    if z not in model_data['Zonal']:
        model_data['Zonal'][z] = {}

    # Initialize or reset the Electrolyser dictionary directly
    model_data['Zonal'][z]['Electrolyser'] = {}
    zonal_elec = model_data['Zonal'][z]['Electrolyser']

    # Source data from capacities.json
    source_techs = capacities.get(z, {}).get('Electrolyser', {})

    for raw_type, clean_type in type_mapping.items():
        # Create sub-dictionary for this electrolyser type
        zonal_elec[clean_type] = {}
        tech_data = source_techs.get(raw_type, {})

        for raw_key, model_key in key_map.items():
            val = tech_data.get(raw_key, 0.0)

            # Extract nested 'Value' if present (standard for capacity.json dictionaries)
            if isinstance(val, dict):
                val = val.get('Value', 0.0)

            try:
                val = float(val)
            except (ValueError, TypeError):
                val = 0.0

            zonal_elec[clean_type][model_key] = val

print(f"✅ Successfully restructured and renamed electrolyser parameters into sub-dictionaries for {len(zones)} zones.")

✅ Successfully restructured and renamed electrolyser parameters into sub-dictionaries for 56 zones.


In [ ]:
import json

# Inspecting the restructured Electrolyser data for a sample zone (DE00)
sample_zone = 'DE00'
elec_structure = model_data['Zonal'][sample_zone].get('Electrolyser', {})

print(f"--- Electrolyser structure for {sample_zone} ---")
print(json.dumps(elec_structure, indent=2))

--- Electrolyser structure for DE00 ---
{
  "el_market": {
    "P_nom": 25936.33,
    "number_of_units": 7.0,
    "efficiency": 0.68,
    "H2_storage_GWh": 0.0,
    "ramp_up": 0.0,
    "ramp_down": 0.0,
    "fixed_generation_reduction_pct": 0.0
  },
  "SRES": {
    "P_nom": 920.8100000000001,
    "number_of_units": 7.0,
    "efficiency": 0.68,
    "H2_storage_GWh": 0.0,
    "ramp_up": 0.0,
    "ramp_down": 0.0,
    "fixed_generation_reduction_pct": 0.0
  },
  "DRES": {
    "P_nom": 1000.0,
    "number_of_units": 1.0,
    "efficiency": 0.68,
    "H2_storage_GWh": 0.0,
    "ramp_up": 0.0,
    "ramp_down": 0.0,
    "fixed_generation_reduction_pct": 0.0
  }
}


### SMR

In [ ]:
import pandas as pd

# Define the SMR parameters to extract and their new names in model_data
smr_key_mapping = {
    'CAPACITY [MW]': 'P_nom_MW',
    'HEAT RATE [GJ/MWh]': 'heat_rate_GJ_MWh',
    'VO&M CHARGE [€/MWh]': 'VOM_charge_EUR_MWh',
    'CCS': 'CCS_enabled',
    'MAX RAMP UP [MW/min]': 'ramp_up_MW_min',
    'MAX RAMP DOWN [MW/min]': 'ramp_down_MW_min'
}

# Initialize SMR data for all zones to ensure consistent structure
for zone in zones:
    if zone not in model_data['Zonal']:
        model_data['Zonal'][zone] = {}
    # Initialize the SMR dictionary for the zone
    model_data['Zonal'][zone]['SMR'] = {}

# Process SMR data and map to the appropriate zones
if 'SMR' in locals() and not SMR.empty:
    for smr_node, smr_data in SMR.iterrows():
        smr_country_code = smr_node[:2] # e.g., 'IT' from 'ITh2'
        target_zone = None

        if smr_country_code == 'IT':
            # Special rule for Italy: map to ITN1
            target_zone = 'ITN1'
        else:
            # For other countries, find the matching electricity zone
            matching_zones = [z for z in zones if z.startswith(smr_country_code)]
            if matching_zones:
                target_zone = matching_zones[0]

        if target_zone and target_zone in model_data['Zonal']:
            # Initialize the specific SMR node dictionary
            model_data['Zonal'][target_zone]['SMR'][smr_node] = {}

            # Map SMR parameters directly into the specific SMR node dictionary
            for original_key, model_key in smr_key_mapping.items():
                if original_key in smr_data.index:
                    val = smr_data[original_key]
                    # Ensure CCS is stored as a boolean
                    if model_key == 'CCS_enabled':
                        model_data['Zonal'][target_zone]['SMR'][smr_node][model_key] = bool(val)
                    else:
                        try:
                            model_data['Zonal'][target_zone]['SMR'][smr_node][model_key] = float(val)
                        except (ValueError, TypeError):
                            model_data['Zonal'][target_zone]['SMR'][smr_node][model_key] = 0.0
            print(f"    Mapped SMR {smr_node} to zone {target_zone}.")
        else:
            print(f"    Warning: No matching zone found or target zone invalid for SMR {smr_node}.")
else:
    print("    SMR DataFrame is not defined or is empty, no SMR data mapped.")

print("\n✅ Successfully mapped SMR parameters to model_data['Zonal'][zone]['SMR'][smr_node].")


    Mapped SMR BEh2 to zone BE00.
    Mapped SMR DEh2Z1 to zone DE00.
    Mapped SMR DEh2 to zone DE00.
    Mapped SMR FRh2 to zone FR00.
    Mapped SMR ITh2 to zone ITN1.
    Mapped SMR LTh2Z1 to zone LT00.
    Mapped SMR NLh2 to zone NL00.
    Mapped SMR PTh2Z1 to zone PT00.
    Mapped SMR UKh2 to zone UK00.
    Mapped SMR FIh2 to zone FI00.
    Mapped SMR ALh2 to zone AL00.
    Mapped SMR ATh2 to zone AT00.
    Mapped SMR BAh2 to zone BA00.
    Mapped SMR BGh2 to zone BG00.
    Mapped SMR CHh2 to zone CH00.
    Mapped SMR CYh2 to zone CY00.
    Mapped SMR CZh2 to zone CZ00.
    Mapped SMR DKh2 to zone DKE1.
    Mapped SMR EEh2 to zone EE00.
    Mapped SMR ESh2 to zone ES00.
    Mapped SMR FIh2Al to zone FI00.
    Mapped SMR FIh2N to zone FI00.
    Mapped SMR FIh2S to zone FI00.
    Mapped SMR FRh2N to zone FR00.
    Mapped SMR FRh2S to zone FR00.
    Mapped SMR FRh2SW to zone FR00.
    Mapped SMR GRh2 to zone GR00.
    Mapped SMR HRh2 to zone HR00.
    Mapped SMR HUh2 to zone HU00.


Let's inspect the `model_data['Zonal']` structure for a sample zone to confirm that SMR parameters are correctly nested under the 'SMR' key.

### H2 storage

In [ ]:
import pandas as pd

# Define the H2 storage parameters to extract and their new names in model_data
h2_storage_param_mapping = {
    'CAPACITY [GWh]': 'energy_MWh', # Will be converted from GWh to MWh
    'MAX POWER [MW]': 'p_max_MW', # Discharge power
    'MAX LOAD [MW]': 'p_load_MW',  # Charge power
    'CHARGE EFFICIENCY [%]': 'charge_efficiency_pct',
    'DISCHARGE EFFICIENCY [%]': 'discharge_efficiency_pct',
    'Initial SoC [GWh]': 'initial_soc_MWh' # Will be converted from GWh to MWh
}

# Initialize H2_storage data for all zones to ensure consistent structure
for zone in zones:
    if zone not in model_data['Zonal']:
        model_data['Zonal'][zone] = {}
    model_data['Zonal'][zone]['H2_storage'] = {}

# Process H2_storage data and map to the appropriate zones
if 'H2_storage' in locals() and not H2_storage.empty:
    for idx, row in H2_storage.iterrows():
        h2_node = row['NODE']
        h2_zone_type = row.get('H2 ZONE', 'Zone 2')
        flexibility = row.get('Flexibility', 'Daily')
        country_code = h2_node[:2]

        target_electricity_zone = None

        if country_code == 'IT':
            # Italy specific mapping
            target_electricity_zone = 'ITN1'
        else:
            # Match based on prefix
            matching_zones = [z for z in zones if z.startswith(country_code)]
            if matching_zones:
                target_electricity_zone = matching_zones[0]
                if len(matching_zones) > 1:
                    print(f"    Warning: Multiple zones for {country_code}. Mapping {h2_node} to {target_electricity_zone}.")

        if target_electricity_zone and target_electricity_zone in model_data['Zonal']:
            # Use the node as the unique storage ID
            storage_id = h2_node

            if storage_id not in model_data['Zonal'][target_electricity_zone]['H2_storage']:
                model_data['Zonal'][target_electricity_zone]['H2_storage'][storage_id] = {}

            target_dict = model_data['Zonal'][target_electricity_zone]['H2_storage'][storage_id]
            target_dict['flexibility'] = flexibility

            # Map and convert units
            for original_col, model_key in h2_storage_param_mapping.items():
                val = row.get(original_col, 0.0)

                if '[GWh]' in original_col:
                    val = float(val) * 1e3 # GWh to MWh
                elif '[%]' in original_col:
                    val = float(val) / 100.0 # % to fraction
                else:
                    val = float(val)

                target_dict[model_key] = val

            print(f"    Mapped H2 storage {storage_id} to zone {target_electricity_zone}.")

print("\n✅ Successfully mapped H2 storage parameters to model_data['Zonal'].")

    Mapped H2 storage DEh2 to zone DE00.
    Mapped H2 storage DKh2 to zone DKE1.
    Mapped H2 storage ESh2 to zone ES00.
    Mapped H2 storage FRh2 to zone FR00.
    Mapped H2 storage FRh2S to zone FR00.
    Mapped H2 storage FRh2SW to zone FR00.
    Mapped H2 storage ITh2 to zone ITN1.
    Mapped H2 storage NLh2 to zone NL00.
    Mapped H2 storage NLh2 to zone NL00.
    Mapped H2 storage UKh2 to zone UK00.
    Mapped H2 storage ALh2 to zone AL00.
    Mapped H2 storage ATh2 to zone AT00.
    Mapped H2 storage BAh2 to zone BA00.
    Mapped H2 storage BEh2 to zone BE00.
    Mapped H2 storage BGh2 to zone BG00.
    Mapped H2 storage CHh2 to zone CH00.
    Mapped H2 storage CYh2 to zone CY00.
    Mapped H2 storage CZh2 to zone CZ00.
    Mapped H2 storage DEh2Z1 to zone DE00.
    Mapped H2 storage EEh2 to zone EE00.
    Mapped H2 storage FIh2 to zone FI00.
    Mapped H2 storage FIh2Al to zone FI00.
    Mapped H2 storage FIh2N to zone FI00.
    Mapped H2 storage FIh2S to zone FI00.
    Map

In [ ]:
import pandas as pd

# Improved summary of H2 storage data from the Zonal tier
h2_storage_summary = []

for zone in zones:
    z_data = model_data['Zonal'].get(zone, {})
    h2_storage_dict = z_data.get('H2_storage', {})

    for storage_id, storage_params in h2_storage_dict.items():
        # Convert list of flexibilities to string if multiple exist
        flex_value = storage_params.get('flexibility', 'N/A')
        if isinstance(flex_value, list):
            flex_value = ", ".join(flex_value)

        h2_storage_summary.append({
            'Zone': zone,
            'Storage Node': storage_id,
            'Flexibility': flex_value,
            'Energy [MWh]': storage_params.get('energy_MWh', 0),
            'P_max (Discharge) [MW]': storage_params.get('p_max_MW', 0),
            'P_load (Charge) [MW]': storage_params.get('p_load_MW', 0)
        })

# Create and display the formatted DataFrame
if h2_storage_summary:
    df_h2_storage_summary = pd.DataFrame(h2_storage_summary).set_index(['Zone', 'Storage Node'])
    print("--- Hydrogen Storage Parameters (Zonal Tier) ---")
    display(df_h2_storage_summary)
else:
    print("No Hydrogen Storage Parameters mapped yet.")

--- Hydrogen Storage Parameters (Zonal Tier) ---


,,Flexibility,Energy [MWh],P_max (Discharge) [MW],P_load (Charge) [MW]
Zone,Storage Node,,,,
AL00,ALh2,N/A,0.000000e+00,0.000000,0.000000
AT00,ATh2,N/A,0.000000e+00,0.000000,0.000000
BA00,BAh2,N/A,0.000000e+00,0.000000,0.000000
BE00,BEh2,N/A,0.000000e+00,0.000000,0.000000
BG00,BGh2,N/A,0.000000e+00,0.000000,0.000000
CH00,CHh2,N/A,0.000000e+00,0.000000,0.000000
CY00,CYh2,N/A,0.000000e+00,0.000000,0.000000
CZ00,CZh2,N/A,0.000000e+00,0.000000,0.000000
DE00,DEh2,Daily,1.991500e+07,38853.000000,32717.000000


### Battery Parameters

In [ ]:
# Map parameters from df_batteries into model_data['Zonal']
for zone in zones:
    if zone in df_batteries.index:
        # Ensure the zonal container exists
        if zone not in model_data['Zonal']:
            model_data['Zonal'][zone] = {}

        zonal_tier = model_data['Zonal'][zone]

        # Initialize the new Battery structure
        if 'Battery' not in zonal_tier:
            zonal_tier['Battery'] = {'Utility': {}, 'Local': {}}

        # Utility Battery Mapping
        zonal_tier['Battery']['Utility']['P_nom'] = df_batteries.loc[zone, 'Utility_P_nom (MW)']
        zonal_tier['Battery']['Utility']['E_nom'] = df_batteries.loc[zone, 'Utility_E_nom (MWh)']
        zonal_tier['Battery']['Utility']['efficiency'] = df_batteries.loc[zone, 'Utility_Efficiency']
        zonal_tier['Battery']['Utility']['ramp_up'] = 0.0
        zonal_tier['Battery']['Utility']['ramp_down'] = 0.0

        # Residential (Local) Battery Mapping
        zonal_tier['Battery']['Local']['P_nom'] = df_batteries.loc[zone, 'Residential_P_nom (MW)']
        zonal_tier['Battery']['Local']['E_nom'] = df_batteries.loc[zone, 'Residential_E_nom (MWh)']
        zonal_tier['Battery']['Local']['efficiency'] = df_batteries.loc[zone, 'Residential_Efficiency']
        zonal_tier['Battery']['Local']['ramp_up'] = 0.0
        zonal_tier['Battery']['Local']['ramp_down'] = 0.0

        # Clean up old flat keys if they exist
        old_keys = [
            'Battery_Utility_P_nom', 'Battery_Utility_E_nom', 'Battery_Utility_efficiency',
            'Battery_Utility_ramp_up', 'Battery_Utility_ramp_down',
            'Battery_Local_P_nom', 'Battery_Local_E_nom', 'Battery_Local_efficiency',
            'Battery_Local_ramp_up', 'Battery_Local_ramp_down'
        ]
        for old_k in old_keys:
            if old_k in zonal_tier:
                del zonal_tier[old_k]

print(f"✅ Successfully mapped Utility and Local battery parameters to model_data['Zonal'][zone]['Battery'] for {len(zones)} zones.")

# Verification for a sample zone
sample_z = 'ITN1'
print(f"\nSample mapping for {sample_z}:")
if 'Battery' in model_data['Zonal'][sample_z]:
    for sub_cat, params in model_data['Zonal'][sample_z]['Battery'].items():
        print(f"  - {sub_cat}:")
        for k, v in params.items():
            print(f"    - {k}: {v}")


✅ Successfully mapped Utility and Local battery parameters to model_data['Zonal'][zone]['Battery'] for 56 zones.

Sample mapping for ITN1:
  - Utility:
    - P_nom: 1474.347
    - E_nom: 5635.575480000001
    - efficiency: 0.8499999999999999
    - ramp_up: 0.0
    - ramp_down: 0.0
  - Local:
    - P_nom: 4267.300000000001
    - E_nom: 9778.3
    - efficiency: 0.8500000000000003
    - ramp_up: 0.0
    - ramp_down: 0.0


### Thermal / Conventional Generators

In [ ]:
capacities['FR00']['Thermal']

{'Nuclear': 63020.0,
 'Hard coal - old 1': 0.0,
 'Hard coal - old 2': 0.0,
 'Hard coal - new': 0.0,
 'Hard coal - CCS': 0.0,
 'Lignite - old 1': 0.0,
 'Lignite - old 2': 0.0,
 'Lignite - new': 0.0,
 'Lignite - CCS': 0.0,
 'Gas - Conventional old 1': 0.0,
 'Gas - Conventional old 2': 0.0,
 'Gas - CCGT old 1': 0.0,
 'Gas - CCGT old 2': 0.0,
 'Gas - CCGT new': 414.0,
 'Gas - CCGT CCS': 0.0,
 'Gas - OCGT old': 199.0,
 'Gas - OCGT new': 437.0,
 'Light oil': 1331.0,
 'Heavy oil - old 1': 0.0,
 'Heavy oil - old 2': 0.0,
 'Oil shale - old': 0.0,
 'Oil shale - new': 0.0,
 'Gas - CCGT present 1': 792.0,
 'Gas - CCGT present 2': 5347.0,
 'Hydrogen - Fuel Cell': 0.0,
 'Hydrogen - CCGT': 0.0,
 'Hydrogen - OCGT': 0.0}

In [ ]:
# Redefining lists to ensure they are available for aggregation
gas_techs = ['Gas - Conventional old 1', 'Gas - Conventional old 2', 'Gas - CCGT old 1', 'Gas - CCGT old 2', 'Gas - CCGT new', 'Gas - CCGT CCS', 'Gas - OCGT old', 'Gas - OCGT new', 'Gas - CCGT present 1', 'Gas - CCGT present 2']
hard_coal_techs = ['Hard coal - old 1', 'Hard coal - old 2', 'Hard coal - new', 'Hard coal - CCS']
lignite_techs = ['Lignite - old 1', 'Lignite - old 2', 'Lignite - new', 'Lignite - CCS']
oil_techs = ['Light oil', 'Heavy oil - old 1', 'Heavy oil - old 2', 'Oil shale - old', 'Oil shale - new']

thermal_database = {}

for zone in zones:
    thermal_database[zone] = {'Nuclear': 0.0, 'Hard coal': 0.0, 'Lignite': 0.0, 'Gas': 0.0, 'Oil': 0.0, 'Hydrogen_Fuel_Cell': 0.0, 'Hydrogen_CCGT': 0.0, 'Hydrogen_OCGT': 0.0}
    if zone in capacities:
        z_caps = capacities[zone]
        thermal_data = z_caps.get('Thermal', z_caps.get('Conventional', {}))

        if thermal_data:
            for tech in gas_techs: thermal_database[zone]['Gas'] += thermal_data.get(tech, 0.0)
            for tech in hard_coal_techs: thermal_database[zone]['Hard coal'] += thermal_data.get(tech, 0.0)
            for tech in lignite_techs: thermal_database[zone]['Lignite'] += thermal_data.get(tech, 0.0)
            for tech in oil_techs: thermal_database[zone]['Oil'] += thermal_data.get(tech, 0.0)
            thermal_database[zone]['Nuclear'] = thermal_data.get('Nuclear', 0.0)
            thermal_database[zone]['Hydrogen_Fuel_Cell'] = thermal_data.get('Hydrogen - Fuel Cell', 0.0)
            thermal_database[zone]['Hydrogen_CCGT'] = thermal_data.get('Hydrogen - CCGT', 0.0)
            thermal_database[zone]['Hydrogen_OCGT'] = thermal_data.get('Hydrogen - OCGT', 0.0)


In [ ]:
gas_techs = ['Gas - Conventional old 1', 'Gas - Conventional old 2', 'Gas - CCGT old 1', 'Gas - CCGT old 2', 'Gas - CCGT new', 'Gas - CCGT CCS', 'Gas - OCGT old', 'Gas - OCGT new', 'Gas - CCGT present 1', 'Gas - CCGT present 2']
hard_coal_techs = ['Hard coal - old 1', 'Hard coal - old 2', 'Hard coal - new', 'Hard coal - CCS']
lignite_techs = ['Lignite - old 1', 'Lignite - old 2', 'Lignite - new', 'Lignite - CCS']
oil_techs = ['Light oil', 'Heavy oil - old 1', 'Heavy oil - old 2', 'Oil shale - old', 'Oil shale - new']

thermal_database = {}

for zone in zones:
    thermal_database[zone] = {'Nuclear': 0.0, 'Hard coal': 0.0, 'Lignite': 0.0, 'Gas': 0.0, 'Oil': 0.0, 'Hydrogen_Fuel_Cell': 0.0, 'Hydrogen_CCGT': 0.0, 'Hydrogen_OCGT': 0.0}
    if zone in capacities:
        z_caps = capacities[zone]
        thermal_data = z_caps.get('Thermal', z_caps.get('Conventional', {}))

        if thermal_data:
            for tech in gas_techs: thermal_database[zone]['Gas'] += thermal_data.get(tech, 0.0)
            for tech in hard_coal_techs: thermal_database[zone]['Hard coal'] += thermal_data.get(tech, 0.0)
            for tech in lignite_techs: thermal_database[zone]['Lignite'] += thermal_data.get(tech, 0.0)
            for tech in oil_techs: thermal_database[zone]['Oil'] += thermal_data.get(tech, 0.0)
            thermal_database[zone]['Nuclear'] = thermal_data.get('Nuclear', 0.0)
            thermal_database[zone]['Hydrogen_Fuel_Cell'] = thermal_data.get('Hydrogen - Fuel Cell', 0.0)
            thermal_database[zone]['Hydrogen_CCGT'] = thermal_data.get('Hydrogen - CCGT', 0.0)
            thermal_database[zone]['Hydrogen_OCGT'] = thermal_data.get('Hydrogen - OCGT', 0.0)

# Map the aggregated database to model_data['Zonal']
for zone in zones:
    if zone in thermal_database:
        z_map = model_data['Zonal'][zone]
        # Initialize the 'Thermal' sub-dictionary if it doesn't exist
        if 'Thermal' not in z_map:
            z_map['Thermal'] = {}
        data = thermal_database[zone]
        z_map['Thermal']['Nuclear_P_nom'] = data['Nuclear']
        z_map['Thermal']['Coal_P_nom'] = data['Hard coal']
        z_map['Thermal']['Lignite_P_nom'] = data['Lignite']
        z_map['Thermal']['Gas_P_nom'] = data['Gas']
        z_map['Thermal']['Oil_P_nom'] = data['Oil']
        z_map['Thermal']['H2_FuelCell_P_nom'] = data['Hydrogen_Fuel_Cell']
        z_map['Thermal']['H2_CCGT_P_nom'] = data['Hydrogen_CCGT']
        z_map['Thermal']['H2_OCGT_P_nom'] = data['Hydrogen_OCGT']

print("✅ Aggregated thermal capacities mapped to model_data['Zonal']['zone']['Thermal'] for all zones.")


✅ Aggregated thermal capacities mapped to model_data['Zonal']['zone']['Thermal'] for all zones.


### Solar and Wind Technolgies into model_data

In [ ]:
for z in zones:
    # Standardize PV_capacities (Already done, but ensuring consistency in this loop block)
    if 'SRES_z1' in PV_capacities[z]:
        del PV_capacities[z]['SRES_z1']
    if 'SRES_z2' in PV_capacities[z]:
        PV_capacities[z]['SRES'] = PV_capacities[z].pop('SRES_z2')

    # Standardize Wind_Onshore_capacities
    if 'SRES_z1' in Wind_Onshore_capacities[z]:
        del Wind_Onshore_capacities[z]['SRES_z1']
    if 'SRES_z2' in Wind_Onshore_capacities[z]:
        Wind_Onshore_capacities[z]['SRES'] = Wind_Onshore_capacities[z].pop('SRES_z2')

    # Standardize Wind_Offshore_capacities
    if 'SRES_z1' in Wind_Offshore_capacities[z]:
        del Wind_Offshore_capacities[z]['SRES_z1']
    if 'SRES_z2' in Wind_Offshore_capacities[z]:
        Wind_Offshore_capacities[z]['SRES'] = Wind_Offshore_capacities[z].pop('SRES_z2')

# Verification for AL00
print(f"Updated PV keys for AL00: {list(PV_capacities['AL00'].keys())}")
print(f"Updated Wind Onshore keys for AL00: {list(Wind_Onshore_capacities['AL00'].keys())}")
print(f"Updated Wind Offshore keys for AL00: {list(Wind_Offshore_capacities['AL00'].keys())}")

Updated PV keys for AL00: ['Total', 'El_market', 'DRES', 'SRES']
Updated Wind Onshore keys for AL00: ['Total', 'El_market', 'DRES', 'SRES']
Updated Wind Offshore keys for AL00: ['Total', 'El_market', 'DRES', 'SRES']


In [ ]:
for z in zones:
    # Ensure the zonal container exists
    if z not in model_data['Zonal']:
        model_data['Zonal'][z] = {}

    zonal_tier = model_data['Zonal'][z]

    # Initialize nested dictionaries
    zonal_tier['PV'] = {'capacities': {}, 'profiles': {}}
    zonal_tier['Wind_Onshore'] = {'capacities': {}, 'profiles': {}}
    zonal_tier['Wind_Offshore'] = {'capacities': {}, 'profiles': {}}

    # 1. Map Standardized Capacities from previously prepared and cleaned dictionaries
    zonal_tier['PV']['capacities'] = PV_capacities.get(z, {})
    zonal_tier['Wind_Onshore']['capacities'] = Wind_Onshore_capacities.get(z, {})
    zonal_tier['Wind_Offshore']['capacities'] = Wind_Offshore_capacities.get(z, {})

    # 2. Map Profiles
    pv_col = f'{z}_Utility_PV'
    zonal_tier['PV']['profiles']['generation'] = RES_profiles[pv_col] if pv_col in RES_profiles.columns else 0.0

    won_col = f'{z}_Wind_Onshore'
    zonal_tier['Wind_Onshore']['profiles']['generation'] = RES_profiles[won_col] if won_col in RES_profiles.columns else 0.0

    woff_col = f'{z}_Wind_Offshore'
    zonal_tier['Wind_Offshore']['profiles']['generation'] = RES_profiles[woff_col] if woff_col in RES_profiles.columns else 0.0

print(f"✅ Successfully mapped standardized PV, Wind_Onshore, and Wind_Offshore to model_data['Zonal'] for {len(zones)} zones.")
# Final Verification for a sample zone
print(f"Final keys for {zones[0]} PV: {list(model_data['Zonal'][zones[0]]['PV']['capacities'].keys())}")

✅ Successfully mapped standardized PV, Wind_Onshore, and Wind_Offshore to model_data['Zonal'] for 56 zones.
Final keys for AL00 PV: ['Total', 'El_market', 'DRES', 'SRES']


### Rooftop PV data

In [ ]:
for z in zones:
    # Ensure the zonal container exists
    if z not in model_data['Zonal']:
        model_data['Zonal'][z] = {}

    zonal_tier = model_data['Zonal'][z]

    # Initialize nested dictionary for Rooftop PV
    zonal_tier['Rooftop_PV'] = {'capacities': {}, 'profiles': {}}

    # 1. Map Capacities (Convert GW to MW)
    solar_cap_data = capacities.get(z, {}).get('Solar', {})
    cap_val = solar_cap_data.get('Installed capacities Rooftop (GW):', 0.0)

    # Handle if the value is a nested dict like {'Value': 0}
    if isinstance(cap_val, dict):
        cap_val = cap_val.get('Value', 0.0)

    zonal_tier['Rooftop_PV']['capacities']['P_nom'] = float(cap_val) * 1e3 # Convert GW to MW

    # 2. Map Profiles
    rooftop_col = f'{z}_Rooftop_PV'
    if rooftop_col in RES_profiles.columns:
        zonal_tier['Rooftop_PV']['profiles']['generation'] = RES_profiles[rooftop_col]
    else:
        zonal_tier['Rooftop_PV']['profiles']['generation'] = 0.0

    # Remove the old 'Prosumer_generation' key if it was previously set
    if 'Prosumer_generation' in zonal_tier:
        del zonal_tier['Prosumer_generation']

print(f"✅ Successfully mapped nested Rooftop_PV (capacities and profiles) to model_data['Zonal'] for {len(zones)} zones.")


✅ Successfully mapped nested Rooftop_PV (capacities and profiles) to model_data['Zonal'] for 56 zones.


### Other_RES

In [ ]:
for z in zones:
    # Ensure the zonal container exists
    if z not in model_data['Zonal']:
        model_data['Zonal'][z] = {}

    zonal_tier = model_data['Zonal'][z]

    # Initialize nested dictionary for Other_RES
    zonal_tier['Other_RES'] = {'capacities': {}, 'profiles': {}}

    # 1. Map Capacities
    # Note: capacities JSON uses 'Other_Res' with mixed case
    other_res_cap_data = capacities.get(z, {}).get('Other_Res', {})
    cap_val = other_res_cap_data.get('Installed capacity excl. clim.dependent bands (MW):', 0.0)

    # Handle if the value is a nested dict like {'Value': 0} just in case
    if isinstance(cap_val, dict):
        cap_val = cap_val.get('Value', 0.0)

    zonal_tier['Other_RES']['capacities']['P_nom'] = float(cap_val)

    # 2. Map Profiles directly from Other_df
    other_res_col = f'{z}_Other_RES'
    if 'Other_df' in locals() and other_res_col in Other_df.columns:
        zonal_tier['Other_RES']['profiles']['generation'] = Other_df[other_res_col]
    else:
        zonal_tier['Other_RES']['profiles']['generation'] = 0.0

print(f"✅ Successfully mapped nested Other_RES (capacities and profiles) to model_data['Zonal'] for {len(zones)} zones.")


✅ Successfully mapped nested Other_RES (capacities and profiles) to model_data['Zonal'] for 56 zones.


### Other Non RES

In [ ]:
for z in zones:
    # Ensure the zonal container exists
    if z not in model_data['Zonal']:
        model_data['Zonal'][z] = {}

    zonal_tier = model_data['Zonal'][z]

    # Initialize nested dictionary for Other_Non_RES
    zonal_tier['Other_Non_RES'] = {'capacities': {}, 'profiles': {}}

    # 1. Map Capacities
    other_non_res_cap_data = capacities.get(z, {}).get('Other_Non_Res', {})
    cap_val = other_non_res_cap_data.get('Installed capacity (MW)', 0.0)

    # Handle if the value is a nested dict like {'Value': 0} just in case
    if isinstance(cap_val, dict):
        cap_val = cap_val.get('Value', 0.0)

    zonal_tier['Other_Non_RES']['capacities']['P_nom'] = float(cap_val)

    # 2. Map Profiles directly from Other_df
    other_non_res_col = f'{z}_Other_Non_RES'
    if 'Other_df' in locals() and other_non_res_col in Other_df.columns:
        zonal_tier['Other_Non_RES']['profiles']['generation'] = Other_df[other_non_res_col]
    else:
        zonal_tier['Other_Non_RES']['profiles']['generation'] = 0.0

print(f"✅ Successfully mapped nested Other_Non_RES (capacities and profiles) to model_data['Zonal'] for {len(zones)} zones.")


✅ Successfully mapped nested Other_Non_RES (capacities and profiles) to model_data['Zonal'] for 56 zones.


### CSP

In [ ]:
ST_nostorage_zones = []
ST_storage_zones = []

for zone, data in capacities.items():
    solar_data = data.get('Solar', {})

    # 1. Thermal Solar (General/No storage specified)
    thermal_val = solar_data.get('Installed capacities Thermal Solar (GW):', 0.0)
    val_ns = thermal_val.get('Value', 0.0) if isinstance(thermal_val, dict) else thermal_val
    if val_ns > 0:
        ST_nostorage_zones.append(zone)

    # 2. Solar Thermal with Storage
    stws_val = solar_data.get('Installed capacities Solar Thermal with Storage (GW):', 0.0)
    val_s = stws_val.get('Value', 0.0) if isinstance(stws_val, dict) else stws_val
    if val_s > 0:
        ST_storage_zones.append(zone)

print(f"Zones with Thermal Solar (No Storage specified): {ST_nostorage_zones}")
print(f"Zones with Solar Thermal with Storage: {ST_storage_zones}")

Zones with Thermal Solar (No Storage specified): ['ES00', 'ITS1', 'ITSI', 'LV00']
Zones with Solar Thermal with Storage: ['ES00']


In [ ]:
import pandas as pd

# Identify columns matching the CSP suffix in RES_profiles
csp_columns = [col for col in RES_profiles.columns if col.endswith('_CSP')]

# Extract zone names from column headers (e.g., 'ES00_CSP' -> 'ES00')
csp_zones_in_profiles = [col.split('_')[0] for col in csp_columns]

print(f"Number of zones with CSP profiles in RES_profiles: {len(csp_zones_in_profiles)}")
if csp_zones_in_profiles:
    print(f"Zones found: {csp_zones_in_profiles}")

Number of zones with CSP profiles in RES_profiles: 4
Zones found: ['ES00', 'ITS1', 'ITSI', 'LV00']


In [ ]:
import pandas as pd

# Map Solar Thermal No Storage (nostorage)
for z in ST_nostorage_zones:
    if z not in model_data['Zonal']:
        model_data['Zonal'][z] = {}
    if 'Solar_thermal' not in model_data['Zonal'][z]:
        model_data['Zonal'][z]['Solar_thermal'] = {}

    # Ensure nostorage structure exists
    model_data['Zonal'][z]['Solar_thermal']['nostorage'] = {'capacities': {}, 'profiles': {}}

    st_cap_data = capacities.get(z, {}).get('Solar', {})
    cap_val = st_cap_data.get('Installed capacities Thermal Solar (GW):', 0.0)
    if isinstance(cap_val, dict):
        cap_val = cap_val.get('Value', 0.0)

    model_data['Zonal'][z]['Solar_thermal']['nostorage']['capacities']['P_nom'] = float(cap_val) * 1e3

    csp_col = f'{z}_CSP'
    if csp_col in RES_profiles.columns:
        model_data['Zonal'][z]['Solar_thermal']['nostorage']['profiles']['generation'] = RES_profiles[csp_col]
    else:
        model_data['Zonal'][z]['Solar_thermal']['nostorage']['profiles']['generation'] = 0.0

# Map Solar Thermal with Storage (storage)
for z in ST_storage_zones:
    if 'Solar_thermal' not in model_data['Zonal'][z]:
        model_data['Zonal'][z]['Solar_thermal'] = {}

    model_data['Zonal'][z]['Solar_thermal']['storage'] = {'capacities': {}}
    st_cap_data = capacities.get(z, {}).get('Solar', {})

    # Power Capacity
    p_nom = st_cap_data.get('Installed capacities Solar Thermal with Storage (GW):', 0.0)
    if isinstance(p_nom, dict): p_nom = p_nom.get('Value', 0.0)

    # Storage Capacity
    e_nom = st_cap_data.get('Storage capacities Solar Thermal with Storage (GWh):', 0.0)
    if isinstance(e_nom, dict): e_nom = e_nom.get('Value', 0.0)

    model_data['Zonal'][z]['Solar_thermal']['storage']['capacities']['P_nom'] = float(p_nom) * 1e3
    model_data['Zonal'][z]['Solar_thermal']['storage']['capacities']['E_nom'] = float(e_nom) * 1e3

print(f"✅ Successfully mapped Solar_thermal for {len(ST_nostorage_zones)} zones (nostorage) and {len(ST_storage_zones)} zones (storage).")

✅ Successfully mapped Solar_thermal for 4 zones (nostorage) and 1 zones (storage).


# Saving Model Data and Input data

In [ ]:
import os
import json
import pandas as pd
import numpy as np

# Create the directory if it doesn't exist
if not os.path.exists(MODEL_DATA_DIR):
    os.makedirs(MODEL_DATA_DIR)
    print(f'Created directory: {MODEL_DATA_DIR}')

# Helper to handle pandas objects
def pd_encoder(obj):
    if isinstance(obj, pd.Series):
        return {str(k): v for k, v in obj.to_dict().items()}
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient='records')
    if isinstance(obj, (pd.Timestamp, np.datetime64)):
        return str(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return str(obj)

def stringify_keys(d):
    """Recursively convert dictionary keys to strings to ensure JSON compatibility."""
    if isinstance(d, dict):
        return {str(k): stringify_keys(v) for k, v in d.items()}
    elif isinstance(d, list):
        return [stringify_keys(i) for i in d]
    return d

model_data_file_path = os.path.join(MODEL_DATA_DIR, 'model_data.json')

try:
    # We first stringify keys to avoid the Timestamp key error
    clean_data = stringify_keys(model_data)
    with open(model_data_file_path, 'w', encoding='utf-8') as f:
        json.dump(clean_data, f, default=pd_encoder)
        f.flush()
        os.fsync(f.fileno())
    print(f'✅ Successfully saved consolidated model_data to: {model_data_file_path}')
    print(f'File size check: {os.path.getsize(model_data_file_path)} bytes')
except Exception as e:
    print(f'❌ Failed to save consolidated model_data: {e}')

✅ Successfully saved consolidated model_data to: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/model_data/Europe/DE/2035/profile_3/model_data.json
File size check: 294532148 bytes


In [ ]:
print("--- Top Level Layers ---")
print(model_data.keys())

print("\n--- Europe Layer Keys ---")
print(model_data['Europe'].keys())

print("\n--- Regional Layer Keys ---")
print(model_data['Regional'].keys())

print("\n--- Zonal Layer (Zones) ---")
print(list(model_data['Zonal'].keys()))

# Inspect a sample zone's internal structure
sample_z = zones[0]
print(f"\n--- Sample Zonal Keys ({sample_z}) ---")
print(list(model_data['Zonal'][sample_z].keys()))

--- Top Level Layers ---
dict_keys(['Europe', 'Regional', 'Zonal', 'Local'])

--- Europe Layer Keys ---
dict_keys(['prices', 'CommonData', 'eFuel'])

--- Regional Layer Keys ---
dict_keys(['electricity', 'H2', 'CH4'])

--- Zonal Layer (Zones) ---
['AL00', 'AT00', 'BA00', 'BE00', 'BG00', 'CH00', 'CY00', 'CZ00', 'DE00', 'DKE1', 'DKW1', 'EE00', 'ES00', 'FI00', 'FR00', 'GR00', 'GR03', 'HR00', 'HU00', 'IE00', 'ITCA', 'ITCN', 'ITCS', 'ITN1', 'ITS1', 'ITSA', 'ITSI', 'LT00', 'LUF1', 'LUG1', 'LUV1', 'LV00', 'MD00', 'ME00', 'MK00', 'MT00', 'NL00', 'NOM1', 'NON1', 'NOS1', 'NOS2', 'NOS3', 'PL00', 'PT00', 'RO00', 'RS00', 'SE01', 'SE02', 'SE03', 'SE04', 'SI00', 'SK00', 'TR00', 'UA00', 'UK00', 'UKNI']

--- Sample Zonal Keys (AL00) ---
['Demand', 'H2_demand', 'hydro', 'Electrolyser', 'SMR', 'H2_storage', 'Battery', 'Thermal', 'PV', 'Wind_Onshore', 'Wind_Offshore', 'Rooftop_PV', 'Other_RES', 'Other_Non_RES']


In [ ]:
print("--- Regional Hydrogen Data Skeleton ---")
if 'H2' in model_data.get('Regional', {}):
    for k, v in model_data['Regional']['H2'].items():
        print(f"model_data['Regional']['H2'] -> '{k}' ({type(v).__name__})")
        if isinstance(v, dict):
            print(f"  Keys: {list(v.keys())}")

print("\n--- Zonal Hydrogen Data Skeleton (Sample: ITN1) ---")
sample_zone = 'ITN1'
if sample_zone in model_data.get('Zonal', {}):
    z_data = model_data['Zonal'][sample_zone]

    # 1. Hydrogen Demand Profiles
    print(f"model_data['Zonal']['{sample_zone}']['Hydrogen'] (Demand Profiles):")
    h2_demand = z_data.get('Hydrogen', {})
    if h2_demand:
        for k, v in h2_demand.items():
            print(f"  - '{k}' ({type(v).__name__})")
    else:
        print("  - (Empty)")

    # 2. Hydrogen-related specific parameters (Storage, SMR, Electrolyser, etc.)
    print(f"\nmodel_data['Zonal']['{sample_zone}'] -> Hydrogen & Tech Parameters:")
    h2_keywords = ['H2', 'SMR', 'Electrolyser', 'Hydrogen']
    h2_keys = [k for k in z_data.keys() if any(kw in k for kw in h2_keywords)]

    for k in h2_keys:
        val = z_data[k]
        # Print shape if it's a pandas Series, otherwise print the type
        if hasattr(val, 'shape'):
            print(f"  - '{k}': {type(val).__name__} {val.shape}")
        else:
            print(f"  - '{k}': {type(val).__name__}")
else:
    print(f"Sample zone {sample_zone} not found in model_data['Zonal'].")

--- Regional Hydrogen Data Skeleton ---
model_data['Regional']['H2'] -> 'NTC' (dict)
  Keys: ['internal', 'ammonia', 'bottlenecks', 'import', 'export', 'ext_ext', 'offshore']
model_data['Regional']['H2'] -> 'add_Import' (dict)
  Keys: ['pipeline', 'ammonia', 'profile']
model_data['Regional']['H2'] -> 'Bottlenecks' (list)
model_data['Regional']['H2'] -> 'Electrolysers' (dict)
  Keys: ['DKB2h2_OFF', 'DKHEh2_OFF', 'DKK2h2_OFF', 'DEh2_OFF', 'NL0Bh2_OFF', 'NL0Ch2_OFF', 'NL0Dh2_OFF', 'NL0Eh2_OFF', 'NL0Fh2_OFF', 'NL0Gh2_OFF', 'NL0Jh2_OFF', 'NL0Kh2_OFF', 'NL0Lh2_OFF', 'NL0Mh2_OFF', 'NL0Nh2_OFF', 'NL0Ph2_OFF', 'NL0Qh2_OFF', 'NL0Rh2_OFF', 'NL0Sh2_OFF', 'NL0Th2_OFF', 'NL0Uh2_OFF', 'NL0Vh2_OFF', 'NL0Wh2_OFF', 'NL0Xh2_OFF', 'NL0Yh2_OFF', 'NLLLh2_OFF', 'DKKF_OFF', 'DKKA_OFF', 'DKN1_OFF', 'DKN2_OFF', 'DKN3_OFF', 'DKN4_OFF', 'DKN5_OFF', 'DKN6_OFF', 'DKN7_OFF', 'DKN8_OFF', 'DKN9_OFF', 'DKNS_OFF']

--- Zonal Hydrogen Data Skeleton (Sample: ITN1) ---
model_data['Zonal']['ITN1']['Hydrogen'] (Demand Profil

# Observations